# V10.6 — Temporal repetition and bounded training

Perbaikan eksperimental dari V10.5; hasil recovery baru belum diukur.

- 16 frame/payload, 2 siklus slot, loss gabungan dengan rata-rata logits.
- Forward quantization sesuai evaluasi, gradient straight-through untuk embedder.
- Evaluasi memakai frame berurutan pada FPS asli (V10.5 mengambil frame berjauhan).
- Gate per codec, subset validation tetap, 3 tahap kumulatif, early stop 4 round.
- Resume per step dan per pasangan video/codec validasi; tanpa retry/strength sweep otomatis.
- Checkpoint V10.6 terpisah, warm start bobot V10.5. Optimizer dibuat baru.
- Test hanya pada mode evaluate; hasil V10.5 dan V10.6 tidak langsung sebanding karena sampling berubah.


# Temporal Latent Video Watermarking — Validated Pipeline V10.6

Threshold acceptance tetap; lulus hanya setelah seluruh gate terpenuhi.


## Protokol dan batasan

Neural codec menjadi backbone: embedder memodifikasi latent sebelum quantization dan entropy coding. Payload enam karakter a-z/0-5 dikodekan menjadi 30 bit + CRC-6 + terminated convolutional FEC K=7 rate 1/3, menghasilkan codeword 128 bit. Codeword dibagi menjadi 8 slot temporal × 16 bit. Training memakai dua siklus, validation/test hingga delapan siklus pada frame berurutan.

Extractor membaca segmen dan slot-ID dari latent hasil re-encoding. Slot coverage hanya mengukur cakupan penetapan siklik; lihat frame-slot accuracy dan exact recovery untuk menilai kemampuan model. Presence memakai skor model asli; CRC dan kecocokan payload diperiksa terpisah.

Gate per codec mempertahankan recovery ≥80%, PSNR incremental ≥34 dB dan SSIM incremental ≥0.94. Integritas menggunakan perceptual fingerprint terpisah dari watermark; keberhasilan integritas tidak membuktikan recovery watermark.

Backbone `bmshj2018_factorized` adalah learned image codec yang diterapkan per frame, bukan temporal neural video codec. H.264/H.265 dan neural recompression tetap menjadi skenario pengujian. Tingkat CRF dan quality tidak menjamin bitrate setara; gunakan bpp/bitrate saat membandingkan codec.

Notebook diuji sintaks dan logika resume secara lokal; training GPU dan perbaikan metrik belum divalidasi. Preflight tensor/FEC berjalan otomatis di Colab sebelum training.


In [ ]:
# 1. Setup Colab
from google.colab import drive
drive.mount('/content/drive')

!pip install -q kagglehub opencv-python-headless scikit-image pandas matplotlib tqdm compressai pytorch-msssim==1.0.0 "gradio>=4.44,<7"
!apt-get -qq update
!apt-get -qq install -y ffmpeg

print('Dependency siap.')


In [ ]:
# 2. Import dan konfigurasi eksperimen
import binascii
import gc
import hashlib
import json
import math
import os
import random
import re
import struct
import subprocess
import tempfile
import time
from collections import defaultdict
from pathlib import Path

import cv2
import gradio as gr
import kagglehub
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from pytorch_msssim import ssim as differentiable_ssim
from skimage.metrics import peak_signal_noise_ratio, structural_similarity
from tqdm.auto import tqdm

SEED = 42
FRAME_SIZE = (128, 128)
CLIP_FRAMES = 16
PAYLOADS_PER_STEP = 1
MAX_EVAL_FRAMES = 64

# Profil aman untuk Colab dengan runtime terbatas. Nilai ini hanya membatasi satu
# sesi, bukan mengubah total curriculum/checkpoint penelitian.
RUNTIME_PROFILE = 'colab_limited'
CODEC_BATCH_SIZE = 4
MAX_TRAINING_EPOCHS_PER_RUN = 4
MAX_TRAINING_MINUTES_PER_RUN = 45
REUSE_EXISTING_EVAL_BITSTREAMS = True
ALLOW_DIAGNOSTIC_EVALUATION = False
RUN_MODE = 'train'  # 'evaluate': muat best checkpoint tanpa training
WARM_START_V105 = 'best_v105_stage4_5e07c20697.pth'
EARLY_STOP_ROUNDS = 4
EARLY_STOP_MIN_DELTA = 0.005
VALIDATION_VIDEOS_PER_CODEC = 6
CHECKPOINT_EVERY_STEPS = 5

PAYLOAD_BYTES = 6
PAYLOAD_ALPHABET = 'abcdefghijklmnopqrstuvwxyz012345'
PAYLOAD_BITS_PER_CHAR = 5
PAYLOAD_BITS = PAYLOAD_BYTES * PAYLOAD_BITS_PER_CHAR
CRC_BITS = 6
CONV_CONSTRAINT_LENGTH = 7
CONV_TAIL_BITS = CONV_CONSTRAINT_LENGTH - 1
CONV_INPUT_BITS = PAYLOAD_BITS + CRC_BITS + CONV_TAIL_BITS
CONV_RATE = 3
CONV_CODE_BITS = CONV_INPUT_BITS * CONV_RATE
CODE_BITS = 128
CONV_PADDING_BITS = CODE_BITS - CONV_CODE_BITS
CODE_BYTES = CODE_BITS // 8
assert CONV_CODE_BITS == 126 and CONV_PADDING_BITS == 2

TEMPORAL_SLOTS = 8
BITS_PER_SLOT = CODE_BITS // TEMPORAL_SLOTS
FRAME_SYMBOL_BITS = BITS_PER_SLOT + TEMPORAL_SLOTS
assert CODE_BITS % TEMPORAL_SLOTS == 0
assert CLIP_FRAMES % TEMPORAL_SLOTS == 0
TEMPORAL_REPETITIONS = CLIP_FRAMES // TEMPORAL_SLOTS

TARGET_TEXT = 'sabila'
CONTROL_TEXT = 'kontro'
CORE_NEURAL_QUALITY = 5
INTEGRITY_MAX_DISTANCE = 12

MAX_TRAIN_VIDEOS = 80
MAX_VAL_VIDEOS = 12
MAX_TEST_VIDEOS = 12

STEPS_PER_EPOCH = 50
VALIDATE_EVERY = 2
STAGE_PASS_PATIENCE = 2




OUTPUT_ROOT = Path('/content/drive/MyDrive/Video_data/output/v9_2_temporal_latent_validated')
MODEL_DIR = OUTPUT_ROOT / 'models'
BITSTREAM_DIR = OUTPUT_ROOT / 'bitstreams'
REPORT_DIR = OUTPUT_ROOT / 'reports' / 'v10_6'
for directory in (OUTPUT_ROOT, MODEL_DIR, BITSTREAM_DIR, REPORT_DIR):
    directory.mkdir(parents=True, exist_ok=True)

def seed_everything(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

seed_everything()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

KAGGLE_DATASET_SLUG = 'abdallahwagih/ucf101-videos'
DATASET_ROOT = Path(kagglehub.dataset_download(KAGGLE_DATASET_SLUG))
print('Dataset:', DATASET_ROOT)
print(
    f'Runtime profile={RUNTIME_PROFILE} | batch codec={CODEC_BATCH_SIZE} | '
    f'maksimum training per run={MAX_TRAINING_EPOCHS_PER_RUN} epoch/'
    f'{MAX_TRAINING_MINUTES_PER_RUN} menit'
)


In [ ]:
# 3. Discovery dan split anti-leakage
VIDEO_EXTENSIONS = {'.avi'}
all_video_paths = sorted(
    path for path in DATASET_ROOT.rglob('*.avi')
    if path.is_file()
)
if not all_video_paths:
    raise FileNotFoundError(f'Tidak ada video di {DATASET_ROOT}')

def class_name(path):
    match = re.match(r'^v_(.+?)_g\d+_c\d+$', path.stem)
    return match.group(1) if match else path.parent.name

def group_key(path):
    match = re.search(r'_g(\d+)_', path.stem)
    group = match.group(1) if match else path.stem
    return f'{class_name(path)}:{group}'

def has_part(path, expected):
    return expected.lower() in {part.lower() for part in path.relative_to(DATASET_ROOT).parts}

def group_split(paths, fractions=(0.70, 0.15, 0.15), seed=SEED):
    groups = sorted({group_key(p) for p in paths})
    rng = random.Random(seed)
    rng.shuffle(groups)
    n = len(groups)
    n_train = max(1, int(n * fractions[0]))
    n_val = max(1, int(n * fractions[1]))
    train_groups = set(groups[:n_train])
    val_groups = set(groups[n_train:n_train + n_val])
    test_groups = set(groups[n_train + n_val:])
    return (
        [p for p in paths if group_key(p) in train_groups],
        [p for p in paths if group_key(p) in val_groups],
        [p for p in paths if group_key(p) in test_groups],
    )

explicit_train = [p for p in all_video_paths if has_part(p, 'train')]
explicit_test = [p for p in all_video_paths if has_part(p, 'test')]

if not explicit_test:
    raise FileNotFoundError(
        f'Folder test UCF101 tidak ditemukan atau tidak berisi AVI di {DATASET_ROOT}'
    )

if explicit_train and explicit_test:
    # Test bawaan dataset tidak pernah dipakai untuk training atau pemilihan checkpoint.
    train_candidates, val_candidates, _ = group_split(
        explicit_train, fractions=(0.80, 0.20, 0.0)
    )
    test_candidates = explicit_test
    split_source = 'folder train/test dataset + validation berbasis group dari train'
else:
    train_candidates, val_candidates, test_candidates = group_split(all_video_paths)
    split_source = 'group-aware split 70/15/15'

def balanced_sample(paths, limit, seed):
    by_class = defaultdict(list)
    for path in paths:
        by_class[class_name(path)].append(path)
    rng = random.Random(seed)
    for values in by_class.values():
        rng.shuffle(values)
    chosen = []
    classes = sorted(by_class)
    while len(chosen) < min(limit, len(paths)):
        progressed = False
        for cls in classes:
            if by_class[cls] and len(chosen) < limit:
                chosen.append(by_class[cls].pop())
                progressed = True
        if not progressed:
            break
    return chosen

TRAIN_PATHS = balanced_sample(train_candidates, MAX_TRAIN_VIDEOS, SEED + 1)
VAL_PATHS = balanced_sample(val_candidates, MAX_VAL_VIDEOS, SEED + 2)
TEST_PATHS = balanced_sample(test_candidates, MAX_TEST_VIDEOS, SEED + 3)

train_set, val_set, test_set = map(set, (TRAIN_PATHS, VAL_PATHS, TEST_PATHS))
assert train_set.isdisjoint(val_set)
assert train_set.isdisjoint(test_set)
assert val_set.isdisjoint(test_set)
train_groups = {group_key(p) for p in TRAIN_PATHS}
val_groups = {group_key(p) for p in VAL_PATHS}
test_groups = {group_key(p) for p in TEST_PATHS}
assert train_groups.isdisjoint(val_groups)
assert train_groups.isdisjoint(test_groups)
assert val_groups.isdisjoint(test_groups)
assert all(has_part(path, 'test') and path.suffix.lower() == '.avi' for path in TEST_PATHS)

print(f'Ditemukan: {len(all_video_paths)} video AVI')
print('Sumber split:', split_source)
print(f'Train/Val/Test dipakai: {len(TRAIN_PATHS)}/{len(VAL_PATHS)}/{len(TEST_PATHS)}')
print('Final test: hanya AVI dari folder test; train tidak pernah masuk final test.')
print('Kelas train:', sorted({class_name(p) for p in TRAIN_PATHS}))
print('Leakage check: LULUS')


In [ ]:
# 4. I/O video dengan frame count, resolusi, dan durasi yang konsisten
def _resize_rgb(frame_bgr, frame_size=FRAME_SIZE):
    frame = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
    frame = cv2.resize(frame, (frame_size[1], frame_size[0]), interpolation=cv2.INTER_AREA)
    return frame.astype(np.float32) / 255.0

def sample_clip(path, clip_frames=CLIP_FRAMES):
    cap = cv2.VideoCapture(str(path))
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total <= 0:
        cap.release()
        raise ValueError(f'Video tidak dapat dibaca: {path}')
    start = random.randint(0, max(total - clip_frames, 0))
    cap.set(cv2.CAP_PROP_POS_FRAMES, start)
    frames = []
    for _ in range(clip_frames):
        ok, frame = cap.read()
        if not ok:
            break
        frames.append(_resize_rgb(frame))
    cap.release()
    if not frames:
        raise ValueError(f'Tidak ada frame: {path}')
    while len(frames) < clip_frames:
        frames.append(frames[-1].copy())
    return np.stack(frames)

def load_eval_frames(path, max_frames=MAX_EVAL_FRAMES):
    cap = cv2.VideoCapture(str(path))
    fps = cap.get(cv2.CAP_PROP_FPS) or 25.0
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total <= 0:
        cap.release()
        raise ValueError(f'Video tidak dapat dibaca: {path}')
    frames = []
    for _ in range(min(max_frames, total)):
        ok, frame = cap.read()
        if not ok:
            break
        frames.append(_resize_rgb(frame))
    cap.release()
    if not frames:
        raise ValueError(f'Tidak ada frame evaluasi: {path}')
    return np.stack(frames), float(fps)

def frames_to_tensor(frames):
    return torch.from_numpy(np.asarray(frames)).permute(0, 3, 1, 2).float().to(device)

def tensor_to_frames(tensor):
    return tensor.detach().clamp(0, 1).permute(0, 2, 3, 1).cpu().numpy()

def read_encoded_video(path, expected_frames=None):
    """Decode via FFmpeg CLI agar AV1 terbaca saat OpenCV tidak punya decoder AV1."""
    path = Path(path)
    height, width = FRAME_SIZE
    cmd = [
        'ffmpeg', '-hide_banner', '-loglevel', 'error', '-i', str(path),
        '-an', '-vf', f'scale={width}:{height}:flags=area',
        '-f', 'rawvideo', '-pix_fmt', 'rgb24', 'pipe:1',
    ]
    result = subprocess.run(cmd, capture_output=True, timeout=120)
    bytes_per_frame = height * width * 3
    frame_count = len(result.stdout) // bytes_per_frame
    if result.returncode != 0 or frame_count == 0:
        detail = result.stderr.decode(errors='replace').strip()
        raise RuntimeError(f'FFmpeg gagal mendecode {path}: {detail}')
    usable = frame_count * bytes_per_frame
    frames = np.frombuffer(result.stdout[:usable], dtype=np.uint8).reshape(
        frame_count, height, width, 3
    ).astype(np.float32) / 255.0
    if expected_frames is not None:
        frames = frames[:expected_frames]
        if len(frames) < expected_frames:
            frames = np.concatenate([
                frames,
                np.repeat(frames[-1:], expected_frames - len(frames), axis=0),
            ], axis=0)
    return frames
def ffmpeg_encode(frames, output_path, fps, codec, crf):
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    frames_u8 = np.clip(np.asarray(frames) * 255.0, 0, 255).astype(np.uint8)
    height, width = frames_u8.shape[1:3]
    cmd = [
        'ffmpeg', '-hide_banner', '-loglevel', 'error', '-y',
        '-f', 'rawvideo', '-pix_fmt', 'rgb24', '-s', f'{width}x{height}',
        '-r', str(fps), '-i', '-', '-an', '-c:v', codec,
    ]
    if codec == 'libaom-av1':
        cmd += ['-crf', str(crf), '-b:v', '0', '-cpu-used', '8', '-row-mt', '1']
    else:
        cmd += ['-crf', str(crf), '-preset', 'fast']
    cmd += ['-pix_fmt', 'yuv420p', str(output_path)]
    result = subprocess.run(cmd, input=frames_u8.tobytes(), capture_output=True, timeout=120)
    if result.returncode != 0 or not output_path.exists() or output_path.stat().st_size == 0:
        raise RuntimeError(result.stderr.decode(errors='replace'))
    return output_path
print('Video I/O siap, termasuk decode AV1 melalui FFmpeg CLI.')


In [ ]:
# 5. Payload convolutional FEC/CRC-6, temporal interleaving, dan extractor
CONV_GENERATORS = (0o171, 0o133, 0o165)
CONV_STATES = 1 << (CONV_CONSTRAINT_LENGTH - 1)
CONV_STATE_MASK = CONV_STATES - 1
CONV_REGISTER_MASK = (1 << CONV_CONSTRAINT_LENGTH) - 1

def _parity(value):
    return int(int(value).bit_count() & 1)

CONV_NEXT_STATE = np.zeros((CONV_STATES, 2), dtype=np.int16)
CONV_OUTPUT_BITS = np.zeros((CONV_STATES, 2, CONV_RATE), dtype=np.uint8)
for _state in range(CONV_STATES):
    for _input_bit in (0, 1):
        _register = ((_state << 1) | _input_bit) & CONV_REGISTER_MASK
        CONV_NEXT_STATE[_state, _input_bit] = _register & CONV_STATE_MASK
        CONV_OUTPUT_BITS[_state, _input_bit] = [
            _parity(_register & generator) for generator in CONV_GENERATORS
        ]
del _state, _input_bit, _register

def text_to_payload(text):
    text = str(text).lower()
    if len(text) != PAYLOAD_BYTES:
        raise ValueError(f'Teks harus tepat {PAYLOAD_BYTES} karakter')
    invalid = sorted(set(text) - set(PAYLOAD_ALPHABET))
    if invalid:
        raise ValueError(
            f'Karakter tidak didukung: {invalid}. Gunakan a-z atau 0-5.'
        )
    symbols = np.asarray(
        [PAYLOAD_ALPHABET.index(character) for character in text], dtype=np.uint8
    )
    bits = np.asarray([
        (int(symbol) >> shift) & 1
        for symbol in symbols for shift in range(PAYLOAD_BITS_PER_CHAR - 1, -1, -1)
    ], dtype=np.uint8)
    return torch.tensor(bits, dtype=torch.float32, device=device).unsqueeze(0)

def payload_to_text(payload_bits):
    bits = np.asarray(payload_bits, dtype=np.uint8).reshape(-1)[:PAYLOAD_BITS]
    symbols = bits.reshape(PAYLOAD_BYTES, PAYLOAD_BITS_PER_CHAR)
    weights = (1 << np.arange(PAYLOAD_BITS_PER_CHAR - 1, -1, -1)).astype(np.uint8)
    indices = symbols @ weights
    return ''.join(PAYLOAD_ALPHABET[int(index)] for index in indices)

def crc6(payload_bits):
    """CRC-6/CDMA2000-A atas 30 bit payload (polynomial 0x27)."""
    checksum = 0
    for payload_bit in np.asarray(payload_bits, dtype=np.uint8).reshape(-1):
        feedback = ((checksum >> 5) & 1) ^ int(payload_bit)
        checksum = (checksum << 1) & 0x3F
        if feedback:
            checksum ^= 0x27
    return np.asarray([
        (checksum >> shift) & 1 for shift in range(CRC_BITS - 1, -1, -1)
    ], dtype=np.uint8)

def convolutional_encode(message_bits):
    message = np.asarray(message_bits, dtype=np.uint8).reshape(-1)
    if message.size != PAYLOAD_BITS + CRC_BITS:
        raise ValueError('Convolutional encoder memerlukan 36 message bit.')
    terminated = np.concatenate([
        message, np.zeros(CONV_TAIL_BITS, dtype=np.uint8)
    ])
    state = 0
    encoded = np.zeros(CODE_BITS, dtype=np.uint8)
    cursor = 0
    for input_bit in terminated:
        bit = int(input_bit)
        encoded[cursor:cursor + CONV_RATE] = CONV_OUTPUT_BITS[state, bit]
        state = int(CONV_NEXT_STATE[state, bit])
        cursor += CONV_RATE
    if state != 0 or cursor != CONV_CODE_BITS:
        raise RuntimeError('Terminasi convolutional code gagal.')
    return encoded

def payload_to_codeword(payload):
    payload_np = payload.detach().round().to(torch.uint8).cpu().numpy()
    rows = []
    for row in payload_np:
        payload_bits = row[:PAYLOAD_BITS].astype(np.uint8)
        rows.append(convolutional_encode(np.concatenate([
            payload_bits, crc6(payload_bits)
        ])))
    return torch.tensor(np.stack(rows), dtype=torch.float32, device=device)

def viterbi_decode_soft(codeword_probabilities):
    probabilities = np.clip(
        np.asarray(codeword_probabilities, dtype=np.float64).reshape(-1)[:CONV_CODE_BITS]
        .reshape(-1, CONV_RATE),
        1e-6, 1.0 - 1e-6,
    )
    if probabilities.shape[0] != CONV_INPUT_BITS:
        raise ValueError(f'Viterbi memerlukan {CODE_BITS} code probabilities.')

    metrics = np.full(CONV_STATES, np.inf, dtype=np.float64)
    metrics[0] = 0.0
    parents = np.full(
        (CONV_INPUT_BITS, CONV_STATES), -1, dtype=np.int16
    )
    next_states = np.arange(CONV_STATES, dtype=np.int16)
    input_bits = (next_states & 1).astype(np.uint8)
    predecessor_zero = next_states >> 1
    predecessor_one = predecessor_zero | (CONV_STATES >> 1)

    for time_index, symbol_probability in enumerate(probabilities):
        log_probability = np.log(symbol_probability)
        log_inverse = np.log1p(-symbol_probability)
        branch_cost = -np.sum(
            CONV_OUTPUT_BITS * log_probability.reshape(1, 1, CONV_RATE) +
            (1 - CONV_OUTPUT_BITS) * log_inverse.reshape(1, 1, CONV_RATE),
            axis=2,
        )
        candidate_zero = (
            metrics[predecessor_zero] +
            branch_cost[predecessor_zero, input_bits]
        )
        candidate_one = (
            metrics[predecessor_one] +
            branch_cost[predecessor_one, input_bits]
        )
        choose_one = candidate_one < candidate_zero
        metrics = np.where(choose_one, candidate_one, candidate_zero)
        parents[time_index] = np.where(
            choose_one, predecessor_one, predecessor_zero
        )

    # Encoder selalu diterminasi dengan enam bit nol, jadi final state harus nol.
    state = 0
    decoded = np.empty(CONV_INPUT_BITS, dtype=np.uint8)
    for time_index in range(CONV_INPUT_BITS - 1, -1, -1):
        decoded[time_index] = state & 1
        state = int(parents[time_index, state])
    return decoded

def decode_codeword_soft(codeword_probabilities):
    probabilities = np.asarray(
        codeword_probabilities, dtype=np.float64
    ).reshape(-1)[:CODE_BITS]
    decoded = viterbi_decode_soft(probabilities)
    payload_bits = decoded[:PAYLOAD_BITS]
    checksum_bits = decoded[PAYLOAD_BITS:PAYLOAD_BITS + CRC_BITS]
    tail_bits = decoded[PAYLOAD_BITS + CRC_BITS:]
    crc_valid = bool(
        not np.any(tail_bits) and np.array_equal(checksum_bits, crc6(payload_bits))
    )
    decoded_message = decoded[:PAYLOAD_BITS + CRC_BITS]
    reconstructed_codeword = convolutional_encode(decoded_message)
    hard_codeword = (probabilities >= 0.5).astype(np.uint8)
    corrected_bits = int(np.sum(reconstructed_codeword != hard_codeword))
    return {
        'ecc_success': crc_valid,
        'crc_valid': crc_valid,
        'payload_bits': payload_bits,
        'raw_payload_bits': payload_bits.copy(),
        'decoded_text': (
            payload_to_text(payload_bits)
            if crc_valid else '<invalid-crc>'
        ),
        'raw_text': payload_to_text(payload_bits),
        'corrected_symbols': corrected_bits if crc_valid else np.nan,
        'soft_flips': corrected_bits if crc_valid else 0,
    }

def decode_codeword(codeword_bits):
    bits = np.asarray(codeword_bits, dtype=np.uint8).reshape(-1)[:CODE_BITS]
    probabilities = np.where(bits > 0, 1.0 - 1e-4, 1e-4)
    return decode_codeword_soft(probabilities)

def temporal_frame_targets(codeword, frames_per_payload):
    """Ubah [B,128] menjadi [B*T,16 bit + 8 slot-ID]."""
    batch = codeword.shape[0]
    slots = torch.arange(frames_per_payload, device=codeword.device) % TEMPORAL_SLOTS
    code_segments = codeword.view(batch, TEMPORAL_SLOTS, BITS_PER_SLOT)
    segments = code_segments[:, slots, :]
    slot_targets = slots.unsqueeze(0).expand(batch, -1)
    slot_one_hot = F.one_hot(slot_targets, TEMPORAL_SLOTS).float()
    symbols = torch.cat([segments, slot_one_hot], dim=-1)
    return (
        symbols.reshape(batch * frames_per_payload, FRAME_SYMBOL_BITS),
        segments.reshape(batch * frames_per_payload, BITS_PER_SLOT),
        slot_targets.reshape(batch * frames_per_payload),
    )

def infer_temporal_slots(slot_logits):
    """Infer phase siklik global; codec menjaga urutan frame meski kualitas turun."""
    slot_probabilities = torch.softmax(slot_logits, dim=1)
    frame_indices = torch.arange(slot_logits.shape[0], device=slot_logits.device)
    offset_scores = []
    for offset in range(TEMPORAL_SLOTS):
        expected = (frame_indices + offset) % TEMPORAL_SLOTS
        offset_scores.append(
            torch.log(slot_probabilities[frame_indices, expected].clamp_min(1e-8)).mean()
        )
    inferred_offset = int(torch.stack(offset_scores).argmax().item())
    aligned_slots = (frame_indices + inferred_offset) % TEMPORAL_SLOTS
    confidence = float(
        slot_probabilities[frame_indices, aligned_slots].mean().item()
    )
    return aligned_slots, inferred_offset, confidence, slot_probabilities

def aggregate_temporal_predictions(segment_logits, slot_logits, presence_logits):
    """Rakit codeword dengan CRC-aware cyclic phase search.

    Perubahan V10.3:
    - semua 8 kemungkinan phase temporal diuji;
    - frame diberi bobot berdasarkan slot-confidence DAN presence;
    - logits (bukan probability) yang dirata-ratakan agar evidence lemah tetapi
      konsisten tidak hilang;
    - bila ada kandidat dengan CRC valid, kandidat itu diprioritaskan.

    Ini penting pada H.264/H.265 CRF tinggi karena classifier slot dapat memiliki
    confidence rendah walaupun urutan frame codec sebenarnya tetap terjaga.
    """
    slot_probabilities = torch.softmax(slot_logits, dim=1)
    presence_probabilities = torch.sigmoid(presence_logits).reshape(-1)
    frame_indices = torch.arange(segment_logits.shape[0], device=segment_logits.device)

    candidates = []
    for offset in range(TEMPORAL_SLOTS):
        aligned_slots = (frame_indices + offset) % TEMPORAL_SLOTS
        aligned_slot_prob = slot_probabilities[frame_indices, aligned_slots]

        # Jangan biarkan satu frame ber-confidence sangat kecil menguasai rata-rata.
        frame_weights = (
            aligned_slot_prob.clamp_min(0.05) *
            presence_probabilities.clamp_min(0.10)
        ).to(segment_logits.dtype)

        assignment = F.one_hot(aligned_slots, TEMPORAL_SLOTS).to(
            dtype=segment_logits.dtype
        )
        weighted_assignment = assignment * frame_weights.unsqueeze(1)
        slot_mass = weighted_assignment.sum(dim=0).clamp_min(1e-6)

        assembled_logits = (
            weighted_assignment.transpose(0, 1) @ segment_logits
        ) / slot_mass.unsqueeze(1)

        code_probabilities = torch.sigmoid(assembled_logits).reshape(-1)
        code_bits = (code_probabilities >= 0.5).to(torch.uint8)

        # CRC+tail dari convolutional decoder memberikan sinyal phase yang jauh
        # lebih kuat daripada slot classifier sendirian.
        decoded = decode_codeword_soft(code_probabilities.detach().cpu().numpy())
        phase_log_likelihood = torch.log(
            aligned_slot_prob.clamp_min(1e-8)
        ).mean().item()
        bit_confidence = (
            (code_probabilities - 0.5).abs() * 2.0
        ).mean().item()

        candidates.append({
            'offset': offset,
            'aligned_slots': aligned_slots,
            'code_probabilities': code_probabilities,
            'code_bits': code_bits,
            'crc_valid': bool(decoded['crc_valid']),
            'phase_log_likelihood': float(phase_log_likelihood),
            'bit_confidence': float(bit_confidence),
            'slot_confidence': float(aligned_slot_prob.mean().item()),
        })

    crc_candidates = [c for c in candidates if c['crc_valid']]
    if crc_candidates:
        # CRC valid menjadi prioritas utama; jika lebih dari satu kandidat valid,
        # pilih yang evidence bit + slot-nya paling kuat.
        best = max(
            crc_candidates,
            key=lambda c: (
                c['bit_confidence'],
                c['phase_log_likelihood'],
            ),
        )
    else:
        # Fallback aman ketika seluruh phase gagal CRC.
        best = max(
            candidates,
            key=lambda c: (
                c['phase_log_likelihood'],
                c['bit_confidence'],
            ),
        )

    slot_coverage = int(torch.unique(best['aligned_slots']).numel())
    presence = float(presence_probabilities.mean().item())

    return (
        best['code_bits'].cpu().numpy(),
        best['code_probabilities'].detach().cpu().numpy(),
        presence,
        slot_coverage,
        best['slot_confidence'],
    )

def group_count(channels):
    for groups in (8, 4, 2, 1):
        if channels % groups == 0:
            return groups
    return 1

class TemporalCarrierEmbedder(nn.Module):
    def __init__(
        self, latent_channels, latent_shape, symbol_bits=FRAME_SYMBOL_BITS,
        strength=1.20, carrier_scale=0.35,
    ):
        super().__init__()
        self.latent_channels = latent_channels
        self.latent_shape = tuple(latent_shape)
        self.symbol_bits = symbol_bits
        self.strength = strength
        self.carrier_scale = carrier_scale
        carriers = torch.randn(symbol_bits, latent_channels, *self.latent_shape)
        self.carriers = nn.Parameter(carriers)
        self.refine_gain = nn.Parameter(torch.tensor(0.10))
        self.refine = nn.Sequential(
            nn.Conv2d(latent_channels * 2, latent_channels, 3, padding=1),
            nn.GroupNorm(group_count(latent_channels), latent_channels), nn.SiLU(),
            nn.Conv2d(latent_channels, latent_channels, 3, padding=1),
            nn.GroupNorm(group_count(latent_channels), latent_channels), nn.SiLU(),
            nn.Conv2d(latent_channels, latent_channels, 3, padding=1),
        )

    def normalized_carriers(self):
        flat = self.carriers.flatten(1)
        scale = math.sqrt(flat.shape[1]) / flat.norm(dim=1, keepdim=True).clamp_min(1e-6)
        return (flat * scale).view_as(self.carriers)

    def forward(self, latent, frame_symbols):
        segment_symbols = frame_symbols[:, :BITS_PER_SLOT] * 2.0 - 1.0
        slot_one_hot = frame_symbols[:, BITS_PER_SLOT:]
        slot_symbols = slot_one_hot - (1.0 / TEMPORAL_SLOTS)
        centered_symbols = torch.cat([segment_symbols, slot_symbols], dim=1)
        carriers = self.normalized_carriers()
        carrier_field = torch.einsum('bs,schw->bchw', centered_symbols, carriers)
        carrier_field = carrier_field * (self.carrier_scale / math.sqrt(FRAME_SYMBOL_BITS))
        refinement = self.refine(torch.cat([latent, carrier_field], dim=1))
        gain = torch.clamp(self.refine_gain, 0.0, 0.50)
        delta = torch.tanh(carrier_field + gain * refinement)
        return latent + self.strength * delta, delta

class TemporalLatentExtractor(nn.Module):
    def __init__(self, latent_channels):
        super().__init__()
        hidden = max(192, latent_channels)
        self.features = nn.Sequential(
            nn.Conv2d(latent_channels, hidden, 3, padding=1),
            nn.GroupNorm(group_count(hidden), hidden), nn.SiLU(),
            nn.Conv2d(hidden, hidden, 3, padding=1),
            nn.GroupNorm(group_count(hidden), hidden), nn.SiLU(),
            nn.Conv2d(hidden, hidden, 3, padding=1),
            nn.GroupNorm(group_count(hidden), hidden), nn.SiLU(),
            nn.AdaptiveAvgPool2d((4, 4)),
        )
        self.shared = nn.Sequential(
            nn.Flatten(), nn.Linear(hidden * 4 * 4, 512), nn.SiLU(), nn.Dropout(0.02)
        )
        self.segment_head = nn.Linear(512, BITS_PER_SLOT)
        self.slot_head = nn.Linear(512, TEMPORAL_SLOTS)
        self.presence_head = nn.Linear(512, 1)

    def encode_features(self, latent):
        return self.shared(self.features(latent))

    def classify(self, hidden):
        return (
            self.segment_head(hidden), self.slot_head(hidden),
            self.presence_head(hidden).squeeze(1),
        )

    def forward(self, latent):
        return self.classify(self.encode_features(latent))

target_payload = text_to_payload(TARGET_TEXT)
target_codeword = payload_to_codeword(target_payload)
assert decode_codeword(target_codeword.cpu().numpy())['decoded_text'] == TARGET_TEXT
_symbols, _segments, _slots = temporal_frame_targets(target_codeword, TEMPORAL_SLOTS)
_perfect_segment_logits = (_segments * 2.0 - 1.0) * 20.0
_perfect_slot_logits = F.one_hot(_slots, TEMPORAL_SLOTS).float() * 20.0
_assembled, _, _, _coverage, _ = aggregate_temporal_predictions(
    _perfect_segment_logits, _perfect_slot_logits,
    torch.full((TEMPORAL_SLOTS,), 20.0, device=device),
)
assert np.array_equal(
    _assembled, target_codeword.cpu().numpy().reshape(-1).astype(np.uint8)
)
assert _coverage == TEMPORAL_SLOTS
assert decode_codeword(_assembled)['decoded_text'] == TARGET_TEXT
del _symbols, _segments, _slots, _perfect_segment_logits, _perfect_slot_logits, _assembled
print(
    f'Payload {PAYLOAD_BITS} bit → convolutional FEC/CRC-6 {CODE_BITS} bit → '
    f'{TEMPORAL_SLOTS} slot × {BITS_PER_SLOT} bit.'
)


def aggregate_training_logits(segment_logits, payload_count, frames_per_payload):
    """Average corresponding slots across cycles; retain differentiability."""
    if frames_per_payload % TEMPORAL_SLOTS:
        raise ValueError('Clip harus merupakan kelipatan jumlah slot.')
    return segment_logits.reshape(
        payload_count, frames_per_payload // TEMPORAL_SLOTS,
        TEMPORAL_SLOTS, BITS_PER_SLOT,
    ).mean(dim=1).reshape(payload_count, CODE_BITS)


def temporal_preflight():
    # Runs on CPU before expensive training; no dataset or checkpoint needed.
    payload = torch.stack([text_to_payload(TARGET_TEXT).reshape(-1),
                           text_to_payload(CONTROL_TEXT).reshape(-1)]).cpu()
    code = payload_to_codeword(payload).cpu()
    for frames in (8, 16, 64):
        _, segments, slots = temporal_frame_targets(code, frames)
        logits = ((segments * 2 - 1) * 8).clone().requires_grad_(True)
        assembled = aggregate_training_logits(logits, 2, frames)
        assert assembled.shape == code.shape
        assert torch.equal((assembled > 0), code.bool())
        F.binary_cross_entropy_with_logits(assembled, code).backward()
        assert logits.grad is not None and torch.isfinite(logits.grad).all()
        assert (logits.grad.abs().sum(dim=1) > 0).all()
        for expected, probabilities in zip(payload, torch.sigmoid(assembled).detach().numpy()):
            decoded = decode_codeword_soft(probabilities)
            assert decoded['crc_valid'] and np.array_equal(decoded['payload_bits'], expected.numpy())
    print('PREFLIGHT LULUS: FEC, slot repetition 8/16/64 frame, batch 2, gradient ke semua frame.')

temporal_preflight()


In [ ]:
# 6. Neural codec backbone, latent embedding, dan entropy-coded bitstream
from compressai.zoo import bmshj2018_factorized

_neural_models = {}

def get_neural_model(quality):
    quality = int(quality)
    if quality not in _neural_models:
        model = bmshj2018_factorized(
            quality=quality, pretrained=True
        ).to(device).eval()
        model.update(force=True)
        for parameter in model.parameters():
            parameter.requires_grad_(False)
        _neural_models[quality] = model
    return _neural_models[quality]

CORE_CODEC = get_neural_model(CORE_NEURAL_QUALITY)
with torch.inference_mode():
    probe_latent = CORE_CODEC.g_a(torch.zeros(1, 3, *FRAME_SIZE, device=device))
LATENT_CHANNELS = int(probe_latent.shape[1])
LATENT_SHAPE = tuple(map(int, probe_latent.shape[-2:]))

latent_embedder = TemporalCarrierEmbedder(LATENT_CHANNELS, LATENT_SHAPE).to(device)
latent_extractor = TemporalLatentExtractor(LATENT_CHANNELS).to(device)

def quantize_latent_training(latent, noise=None):
    if noise is None:
        noise = torch.empty_like(latent).uniform_(-0.5, 0.5)
    return latent + noise, noise

def quantize_latent_eval(model, latent):
    medians = model.entropy_bottleneck._get_medians()
    return model.entropy_bottleneck.quantize(latent, 'dequantize', medians)

def latent_training_forward(frames, frame_symbols):
    latent = CORE_CODEC.g_a(frames)
    watermarked_latent, delta = latent_embedder(latent, frame_symbols)
    baseline_hat = quantize_latent_eval(CORE_CODEC, latent)
    quantized = quantize_latent_eval(CORE_CODEC, watermarked_latent)
    watermarked_hat = watermarked_latent + (quantized - watermarked_latent).detach()
    baseline_frames = CORE_CODEC.g_s(baseline_hat).clamp(0, 1)
    watermarked_frames = CORE_CODEC.g_s(watermarked_hat).clamp(0, 1)
    return baseline_frames, watermarked_frames, latent, watermarked_latent, delta

def latent_eval_forward(frames, frame_symbols):
    latent = CORE_CODEC.g_a(frames)
    watermarked_latent, delta = latent_embedder(latent, frame_symbols)
    baseline_hat = quantize_latent_eval(CORE_CODEC, latent)
    watermarked_hat = quantize_latent_eval(CORE_CODEC, watermarked_latent)
    baseline_frames = CORE_CODEC.g_s(baseline_hat).clamp(0, 1)
    watermarked_frames = CORE_CODEC.g_s(watermarked_hat).clamp(0, 1)
    return baseline_frames, watermarked_frames, latent, watermarked_latent, delta

def plain_neural_encode(frames, output_path, fps, quality):
    model = get_neural_model(quality)
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    entries = []
    with torch.inference_mode():
        for start in range(0, len(frames), CODEC_BATCH_SIZE):
            x = frames_to_tensor(frames[start:start + CODEC_BATCH_SIZE])
            packed = model.compress(x)
            shape = tuple(map(int, packed['shape']))
            for batch_index in range(x.shape[0]):
                streams = [group[batch_index] for group in packed['strings']]
                entries.append((shape, streams))
    header = json.dumps({
        'codec': 'bmshj2018-factorized', 'quality': int(quality),
        'fps': float(fps), 'frames': len(entries),
        'height': int(frames.shape[1]), 'width': int(frames.shape[2]),
    }).encode('utf-8')
    with output_path.open('wb') as handle:
        handle.write(b'NVC1'); handle.write(struct.pack('<I', len(header))); handle.write(header)
        for shape, streams in entries:
            handle.write(struct.pack('<IIH', shape[0], shape[1], len(streams)))
            for stream in streams:
                handle.write(struct.pack('<I', len(stream))); handle.write(stream)
    return output_path

def plain_neural_decode(input_path):
    input_path = Path(input_path)
    with input_path.open('rb') as handle:
        if handle.read(4) != b'NVC1':
            raise ValueError('Bukan container NVC1')
        header_len = struct.unpack('<I', handle.read(4))[0]
        header = json.loads(handle.read(header_len).decode('utf-8'))
        model = get_neural_model(header['quality'])
        entries = []
        for _ in range(header['frames']):
            h, w, n_streams = struct.unpack('<IIH', handle.read(10))
            streams = []
            for _ in range(n_streams):
                size = struct.unpack('<I', handle.read(4))[0]
                streams.append(handle.read(size))
            entries.append(((h, w), streams))
    frames = []
    with torch.inference_mode():
        for start in range(0, len(entries), CODEC_BATCH_SIZE):
            batch = entries[start:start + CODEC_BATCH_SIZE]
            shape = batch[0][0]
            n_streams = len(batch[0][1])
            strings = [[entry[1][i] for entry in batch] for i in range(n_streams)]
            frames.extend(tensor_to_frames(model.decompress(strings, shape)['x_hat']))
    return np.stack(frames), header

def latent_watermark_encode(frames, output_path, fps, payload):
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    codeword = payload_to_codeword(payload)
    frame_symbols, _, _ = temporal_frame_targets(codeword, len(frames))
    entries = []
    reconstructed = []
    with torch.inference_mode():
        for start in range(0, len(frames), CODEC_BATCH_SIZE):
            x = frames_to_tensor(frames[start:start + CODEC_BATCH_SIZE])
            symbols = frame_symbols[start:start + x.shape[0]]
            latent = CORE_CODEC.g_a(x)
            watermarked_latent, _ = latent_embedder(latent, symbols)
            strings = CORE_CODEC.entropy_bottleneck.compress(watermarked_latent)
            shape = tuple(map(int, watermarked_latent.shape[-2:]))
            latent_hat = CORE_CODEC.entropy_bottleneck.decompress(strings, shape)
            reconstructed.extend(tensor_to_frames(CORE_CODEC.g_s(latent_hat).clamp(0, 1)))
            entries.extend((shape, stream) for stream in strings)
    header = json.dumps({
        'codec': 'bmshj2018-factorized-latent-watermark',
        'quality': CORE_NEURAL_QUALITY, 'fps': float(fps),
        'frames': len(entries), 'height': int(frames.shape[1]),
        'width': int(frames.shape[2]), 'code_bits': CODE_BITS,
        'temporal_slots': TEMPORAL_SLOTS, 'bits_per_slot': BITS_PER_SLOT,
    }).encode('utf-8')
    with output_path.open('wb') as handle:
        handle.write(b'LWM1'); handle.write(struct.pack('<I', len(header))); handle.write(header)
        for shape, stream in entries:
            handle.write(struct.pack('<II', shape[0], shape[1]))
            handle.write(struct.pack('<I', len(stream))); handle.write(stream)
    return np.stack(reconstructed), output_path

def latent_watermark_decode(input_path):
    input_path = Path(input_path)
    with input_path.open('rb') as handle:
        if handle.read(4) != b'LWM1':
            raise ValueError('Bukan container latent watermark LWM1')
        header_len = struct.unpack('<I', handle.read(4))[0]
        header = json.loads(handle.read(header_len).decode('utf-8'))
        entries = []
        for _ in range(header['frames']):
            h, w = struct.unpack('<II', handle.read(8))
            size = struct.unpack('<I', handle.read(4))[0]
            entries.append(((h, w), handle.read(size)))
    frames = []
    with torch.inference_mode():
        for start in range(0, len(entries), CODEC_BATCH_SIZE):
            batch = entries[start:start + CODEC_BATCH_SIZE]
            shape = batch[0][0]
            strings = [entry[1] for entry in batch]
            latent_hat = CORE_CODEC.entropy_bottleneck.decompress(strings, shape)
            frames.extend(tensor_to_frames(CORE_CODEC.g_s(latent_hat).clamp(0, 1)))
    return np.stack(frames), header

print(
    f'Core neural quality={CORE_NEURAL_QUALITY}; latent={LATENT_CHANNELS}x'
    f'{LATENT_SHAPE[0]}x{LATENT_SHAPE[1]}; temporal carrier embedder siap.'
)


In [ ]:
# 7. Robustness attacks dan deterministic temporal curriculum V10.2
def ffmpeg_bpda(x, codec, crf, fps=25.0):
    with tempfile.TemporaryDirectory() as tmp:
        suffix = '.mkv' if codec == 'libaom-av1' else '.mp4'
        path = Path(tmp) / f'roundtrip{suffix}'
        frames = tensor_to_frames(x)
        ffmpeg_encode(frames, path, fps, codec, crf)
        reconstructed = frames_to_tensor(read_encoded_video(path, len(frames)))
    return x + (reconstructed - x).detach()

def neural_attack_differentiable(x, quality):
    model = get_neural_model(quality)
    model.eval()
    latent = model.g_a(x.clamp(0, 1))
    quantized = quantize_latent_eval(model, latent)
    latent_hat = latent + (quantized - latent).detach() if torch.is_grad_enabled() else quantized
    return model.g_s(latent_hat).clamp(0, 1)

def apply_attack(x, specification):
    name, level = specification
    if name == 'identity':
        return x
    if name == 'noise':
        noisy = x + torch.randn_like(x) * level
        quantized = torch.clamp(torch.round(noisy * 255) / 255, 0, 1)
        return x + (quantized - x).detach()
    if name == 'blur':
        return F.avg_pool2d(x, 3, stride=1, padding=1)
    if name == 'resize':
        reduced = F.interpolate(x, scale_factor=float(level), mode='bilinear', align_corners=False)
        return F.interpolate(reduced, size=x.shape[-2:], mode='bilinear', align_corners=False)
    if name == 'h264':
        return ffmpeg_bpda(x, 'libx264', int(level))
    if name == 'h265':
        return ffmpeg_bpda(x, 'libx265', int(level))
    if name == 'av1':
        return ffmpeg_bpda(x, 'libaom-av1', int(level))
    if name == 'neural':
        return neural_attack_differentiable(x, int(level))
    raise ValueError(specification)

def temporal_prediction_from_latent(latent):
    hidden = latent_extractor.encode_features(latent)
    segment_logits, slot_logits, presence_logits = latent_extractor.classify(hidden)
    return aggregate_temporal_predictions(segment_logits, slot_logits, presence_logits)

def temporal_prediction_tensor(frames):
    return temporal_prediction_from_latent(CORE_CODEC.g_a(frames))

def latent_eval_forward_batched(frames, frame_symbols, batch_size=CODEC_BATCH_SIZE):
    collected = [[] for _ in range(5)]
    with torch.inference_mode():
        for start in range(0, frames.shape[0], batch_size):
            stop = start + batch_size
            values = latent_eval_forward(frames[start:stop], frame_symbols[start:stop])
            for bucket, value in zip(collected, values):
                bucket.append(value)
    return tuple(torch.cat(values, dim=0) for values in collected)

TRAINING_STAGES = [{'name': 'moderate_recovery', 'max_epochs': 30, 'min_epochs': 4, 'extract_domain': 'reencoded_rgb', 'strength_start': 0.4, 'strength_end': 0.34, 'lr': 0.0002, 'embedder_lr': 3e-05, 'freeze_embedder': False, 'retry_strength_factor': 0.95, 'attacks': [('h264', 35), ('h265', 35), ('neural', 3)], 'validation_attacks': [('h264', 35), ('h265', 35), ('neural', 3)], 'payload_weight': 6.0, 'logit_margin': 1.5, 'margin_weight': 1.25, 'auxiliary_weight': 1.0, 'presence_weight': 0.4, 'slot_weight': 0.8, 'quality_target_mse': 0.00038, 'quality_target_ssim': 0.94, 'quality_weight': 3.0, 'min_raw_code_bit_acc': 0.78, 'min_post_ecc_exact': 0.8, 'min_crc_valid': 0.8, 'min_slot_accuracy': 0.8, 'min_presence': 0.8, 'min_negative': 0.95, 'min_incremental_psnr': 34.0, 'min_incremental_ssim': 0.94}, {'name': 'neural_strong', 'max_epochs': 30, 'min_epochs': 4, 'extract_domain': 'reencoded_rgb', 'strength_start': 0.4, 'strength_end': 0.34, 'lr': 0.0002, 'embedder_lr': 3e-05, 'freeze_embedder': False, 'retry_strength_factor': 0.95, 'attacks': [('h264', 35), ('h265', 35), ('neural', 3), ('neural', 1)], 'validation_attacks': [('h264', 35), ('h265', 35), ('neural', 3), ('neural', 1)], 'payload_weight': 6.0, 'logit_margin': 1.5, 'margin_weight': 1.25, 'auxiliary_weight': 1.0, 'presence_weight': 0.4, 'slot_weight': 0.8, 'quality_target_mse': 0.00038, 'quality_target_ssim': 0.94, 'quality_weight': 3.0, 'min_raw_code_bit_acc': 0.78, 'min_post_ecc_exact': 0.8, 'min_crc_valid': 0.8, 'min_slot_accuracy': 0.8, 'min_presence': 0.8, 'min_negative': 0.95, 'min_incremental_psnr': 34.0, 'min_incremental_ssim': 0.94}, {'name': 'all_high_compression', 'max_epochs': 30, 'min_epochs': 4, 'extract_domain': 'reencoded_rgb', 'strength_start': 0.4, 'strength_end': 0.34, 'lr': 0.0002, 'embedder_lr': 3e-05, 'freeze_embedder': False, 'retry_strength_factor': 0.95, 'attacks': [('h264', 35), ('h265', 35), ('neural', 3), ('neural', 1), ('h264', 40), ('h265', 40)], 'validation_attacks': [('h264', 35), ('h265', 35), ('neural', 3), ('neural', 1), ('h264', 40), ('h265', 40)], 'payload_weight': 6.0, 'logit_margin': 1.5, 'margin_weight': 1.25, 'auxiliary_weight': 1.0, 'presence_weight': 0.4, 'slot_weight': 0.8, 'quality_target_mse': 0.00038, 'quality_target_ssim': 0.94, 'quality_weight': 3.0, 'min_raw_code_bit_acc': 0.78, 'min_post_ecc_exact': 0.8, 'min_crc_valid': 0.8, 'min_slot_accuracy': 0.8, 'min_presence': 0.8, 'min_negative': 0.95, 'min_incremental_psnr': 34.0, 'min_incremental_ssim': 0.94}]

def stage_passed(stage, metrics):
    return (
        metrics['raw_code_bit_acc'] >= stage['min_raw_code_bit_acc'] and
        metrics['post_ecc_exact_recovery'] >= stage['min_post_ecc_exact'] and
        metrics['crc_valid_rate'] >= stage['min_crc_valid'] and
        metrics['slot_accuracy'] >= stage['min_slot_accuracy'] and
        metrics['presence_tpr'] >= stage['min_presence'] and
        metrics['negative_rejection'] >= stage['min_negative'] and
        metrics['incremental_watermark_psnr'] >= stage['min_incremental_psnr'] and
        metrics['incremental_watermark_ssim'] >= stage['min_incremental_ssim']
    )

VALIDATION_TEMPORAL_FRAMES = MAX_EVAL_FRAMES
STAGE_PRESENCE_THRESHOLD_GRID = np.linspace(0.50, 0.99, 50)

def deterministic_validation_paths(max_videos, validation_round=0):
    """Ambil subset validation yang dapat direproduksi dan bergilir."""
    paths = list(VAL_PATHS)
    if not paths:
        raise RuntimeError('VAL_PATHS kosong.')
    count = min(int(max_videos), len(paths))
    start = (int(validation_round) * count) % len(paths)
    return [paths[(start + offset) % len(paths)] for offset in range(count)]

def deterministic_validation_payload(path, validation_round=0):
    """Payload tetap per video/round agar perubahan metrik berasal dari model."""
    digest = hashlib.sha256(
        f'{SEED}|{Path(path).name}|{int(validation_round)}'.encode()
    ).digest()
    bits = np.unpackbits(np.frombuffer(digest, dtype=np.uint8), bitorder='big')
    bits = bits[:PAYLOAD_BITS].astype(np.float32)
    return torch.tensor(bits, device=device).unsqueeze(0)

def calibrated_stage_presence(positive_scores, negative_scores, min_negative):
    """Kalibrasi threshold tanpa mengubah target negative rejection stage."""
    positives = np.asarray(positive_scores, dtype=np.float64)
    negatives = np.asarray(negative_scores, dtype=np.float64)
    best = None
    for threshold in STAGE_PRESENCE_THRESHOLD_GRID:
        tpr = float(np.mean(positives >= threshold))
        rejection = float(np.mean(negatives < threshold))
        if rejection >= min_negative:
            candidate = (tpr, rejection, -float(threshold))
            if best is None or candidate > best[0]:
                best = (candidate, float(threshold), tpr, rejection)
    if best is None:
        threshold = 0.99
        return (
            threshold,
            float(np.mean(positives >= threshold)),
            float(np.mean(negatives < threshold)),
        )
    _, threshold, tpr, rejection = best
    return threshold, tpr, rejection

def quick_validation(stage, max_videos=4, validation_round=0, paths_override=None):
    latent_embedder.eval(); latent_extractor.eval()
    correct = total = exact = crc_count = samples = 0
    slot_correct = slot_total = 0
    frame_slot_correct = frame_slot_total = 0
    positive_scores, negative_scores, coverage_values = [], [], []
    incremental_psnr, incremental_ssim, overall_psnr = [], [], []
    paths = paths_override if paths_override is not None else deterministic_validation_paths(max_videos, validation_round)
    with torch.no_grad():
        for path in paths:
            frames_np, _ = load_eval_frames(
                path, max_frames=VALIDATION_TEMPORAL_FRAMES
            )
            usable_frames = (len(frames_np) // TEMPORAL_SLOTS) * TEMPORAL_SLOTS
            if usable_frames < TEMPORAL_SLOTS:
                raise RuntimeError(f'Frame validation tidak cukup: {path}')
            original = frames_to_tensor(frames_np[:usable_frames])
            payload = deterministic_validation_payload(path, validation_round)
            codeword = payload_to_codeword(payload)
            frame_symbols, _, slot_targets = temporal_frame_targets(
                codeword, original.shape[0]
            )
            baseline, watermarked, clean_latent, watermarked_latent, _ = latent_eval_forward_batched(
                original, frame_symbols
            )
            inc_mse = F.mse_loss(watermarked, baseline).item()
            incremental_psnr.append(-10.0 * math.log10(max(inc_mse, 1e-12)))
            incremental_ssim.append(differentiable_ssim(
                watermarked, baseline, data_range=1.0, size_average=True
            ).item())
            overall_mse = F.mse_loss(watermarked, original).item()
            overall_psnr.append(-10.0 * math.log10(max(overall_mse, 1e-12)))

            for attack in stage['validation_attacks']:
                if stage['extract_domain'] == 'quantized_latent':
                    positive_latent = quantize_latent_eval(CORE_CODEC, watermarked_latent)
                    negative_latent = quantize_latent_eval(CORE_CODEC, clean_latent)
                else:
                    positive_latent = CORE_CODEC.g_a(apply_attack(watermarked, attack))
                    negative_latent = CORE_CODEC.g_a(apply_attack(baseline, attack))

                positive_hidden = latent_extractor.encode_features(positive_latent)
                negative_hidden = latent_extractor.encode_features(negative_latent)
                segment_logits, slot_logits, presence_logits = latent_extractor.classify(
                    positive_hidden
                )
                _, _, negative_presence_logits = latent_extractor.classify(negative_hidden)
                predicted, probabilities, presence, slot_coverage, _ = aggregate_temporal_predictions(
                    segment_logits, slot_logits, presence_logits
                )
                outcome = decode_codeword_soft(probabilities)
                expected_code = codeword.cpu().numpy().reshape(-1).astype(np.uint8)
                target = payload.cpu().numpy().reshape(-1).astype(np.uint8)
                correct += int((predicted == expected_code).sum()); total += CODE_BITS
                exact_match = bool(
                    outcome['crc_valid'] and
                    np.array_equal(outcome['payload_bits'], target)
                )
                exact += int(exact_match)
                crc_count += int(outcome['crc_valid']); samples += 1
                aligned_slots, _, _, _ = infer_temporal_slots(slot_logits)
                slot_correct += int((aligned_slots == slot_targets).sum().item())
                slot_total += int(slot_targets.numel())
                frame_slot_correct += int(
                    (slot_logits.argmax(dim=1) == slot_targets).sum().item()
                )
                frame_slot_total += int(slot_targets.numel())
                # Presence score berasal dari model; label benar tidak menjadi input skor.
                positive_scores.append(float(presence))
                negative_scores.append(float(
                    torch.sigmoid(negative_presence_logits).mean().item()
                ))
                coverage_values.append(slot_coverage / TEMPORAL_SLOTS)

    threshold, presence_tpr, negative_rejection = calibrated_stage_presence(
        positive_scores, negative_scores, stage['min_negative']
    )
    latent_embedder.train(); latent_extractor.train()
    return {
        '_positive_scores': positive_scores, '_negative_scores': negative_scores,
        'raw_code_bit_acc': correct / max(total, 1),
        'post_ecc_exact_recovery': exact / max(samples, 1),
        'crc_valid_rate': crc_count / max(samples, 1),
        'slot_accuracy': slot_correct / max(slot_total, 1),
        'frame_slot_accuracy': frame_slot_correct / max(frame_slot_total, 1),
        'slot_coverage': float(np.mean(coverage_values)),
        'presence_threshold': threshold,
        'presence_tpr': presence_tpr,
        'negative_rejection': negative_rejection,
        'incremental_watermark_psnr': float(np.mean(incremental_psnr)),
        'incremental_watermark_ssim': float(np.mean(incremental_ssim)),
        'overall_original_to_watermarked_psnr': float(np.mean(overall_psnr)),
    }

print(
    'Temporal curriculum V10.6:', [(s['name'], s['max_epochs']) for s in TRAINING_STAGES],
    f'validation_frames={VALIDATION_TEMPORAL_FRAMES}',
)


In [ ]:
# 8. Training V10.6 — bounded, resumable, codec-specific validation
assert RUN_MODE in ('train', 'evaluate')
assert MAX_TRAINING_EPOCHS_PER_RUN >= 0 and MAX_TRAINING_MINUTES_PER_RUN > 0
config_for_hash = {
    'version': '10.6-consecutive-ste-repetition-v1', 'seed': SEED,
    'frames': CLIP_FRAMES, 'payloads': PAYLOADS_PER_STEP,
    'frame_size': FRAME_SIZE, 'eval_frames': MAX_EVAL_FRAMES,
    'payload_bits': PAYLOAD_BITS, 'slots': TEMPORAL_SLOTS,
    'quality': CORE_NEURAL_QUALITY, 'stages': TRAINING_STAGES,
    'steps': STEPS_PER_EPOCH, 'validate_every': VALIDATE_EVERY,
    'validation_videos': VALIDATION_VIDEOS_PER_CODEC,
    'early_stop': [EARLY_STOP_ROUNDS, EARLY_STOP_MIN_DELTA],
    'pass_patience': STAGE_PASS_PATIENCE,
    'warm_start': WARM_START_V105,
    'splits': {name: [str(p.relative_to(DATASET_ROOT)) for p in paths]
               for name, paths in [('train', TRAIN_PATHS), ('val', VAL_PATHS), ('test', TEST_PATHS)]},
}
config_hash = hashlib.sha256(json.dumps(config_for_hash, sort_keys=True).encode()).hexdigest()[:10]
REPORT_DIR = OUTPUT_ROOT / 'reports' / 'v10_6' / config_hash
REPORT_DIR.mkdir(parents=True, exist_ok=True)
checkpoint_path = MODEL_DIR / f'checkpoint_v106_{config_hash}.pth'

def best_path_for(index):
    return MODEL_DIR / f'best_v106_stage{index + 1}_{config_hash}.pth'

def load_local_checkpoint(path):
    # Only trusted experiment checkpoints from your own Drive.
    return torch.load(path, map_location=device, weights_only=False)

def atomic_save(value, path):
    tmp = path.with_suffix(path.suffix + '.tmp')
    torch.save(value, tmp)
    os.replace(tmp, path)

def configure_stage_trainability(stage):
    freeze_embedder = bool(stage.get('freeze_embedder', False))
    for parameter in latent_embedder.parameters():
        parameter.requires_grad_(not freeze_embedder)
    for parameter in latent_extractor.parameters():
        parameter.requires_grad_(True)
    return freeze_embedder

def make_optimizer(stage):
    freeze_embedder = configure_stage_trainability(stage)
    parameter_groups = [{
        'params': list(latent_extractor.parameters()),
        'lr': stage['lr'],
        'name': 'extractor',
    }]
    if not freeze_embedder:
        parameter_groups.append({
            'params': list(latent_embedder.parameters()),
            'lr': stage.get('embedder_lr', stage['lr']),
            'name': 'embedder',
        })
    return torch.optim.AdamW(parameter_groups, weight_decay=1e-5)



def snapshot(validation=None):
    return {'latent_embedder': latent_embedder.state_dict(),
            'latent_extractor': latent_extractor.state_dict(),
            'latent_strength': float(latent_embedder.strength),
            'global_epoch': state['global_epoch'], 'stage_index': state['stage'],
            'validation': validation or {}, 'config_hash': config_hash}

def save_training():
    atomic_save({**snapshot(), 'state': state,
                 'optimizer': None if optimizer is None else optimizer.state_dict()}, checkpoint_path)

optimizer = None
if checkpoint_path.exists():
    saved = load_local_checkpoint(checkpoint_path)
    if saved['config_hash'] != config_hash:
        raise ValueError('Checkpoint/config mismatch.')
    state = saved['state']
else:
    warm_path = MODEL_DIR / WARM_START_V105
    if not warm_path.is_file():
        raise FileNotFoundError(f'Warm start tidak ditemukan: {warm_path}. Isi WARM_START_V105 dengan nama best checkpoint V10.5 yang benar.')
    saved = load_local_checkpoint(warm_path)
    state = {'stage': 0, 'epoch': 0, 'global_epoch': int(saved.get('global_epoch', 0)),
             'step': 0, 'totals': {}, 'pending_validation': False,
             'validation_jobs': {}, 'best_score': None, 'bad_rounds': 0,
             'pass_streak': 0, 'stopped': False, 'reason': '',
             'records': [], 'history': [], 'codec_history': []}
    saved['optimizer'] = None
latent_embedder.load_state_dict(saved['latent_embedder'], strict=True)
latent_extractor.load_state_dict(saved['latent_extractor'], strict=True)
latent_embedder.strength = float(saved.get('latent_strength', 0.40))
if state['stage'] < len(TRAINING_STAGES):
    optimizer = make_optimizer(TRAINING_STAGES[state['stage']])
    if saved.get('optimizer') is not None:
        optimizer.load_state_dict(saved['optimizer'])
if not checkpoint_path.exists():
    atomic_save(snapshot(saved.get('validation', {})), best_path_for(0))
    save_training()
print(f"V10.6 resume: global {state['global_epoch']}, stage {state['stage'] + 1}, epoch {state['epoch']}, step {state['step']}/{STEPS_PER_EPOCH}")


def train_one_step(stage, step_index):
    # Repeatable after interruption; each step has a distinct random clip/payload/noise.
    seed_everything(SEED + state['global_epoch'] * STEPS_PER_EPOCH + step_index)
    totals = defaultdict(float, state['totals'])
    base_clip = frames_to_tensor(sample_clip(random.choice(TRAIN_PATHS)))
    frames_per_payload = base_clip.shape[0]
    if frames_per_payload != CLIP_FRAMES:
        raise RuntimeError('Jumlah frame clip tidak sesuai konfigurasi.')

    payload = torch.randint(0, 2, (PAYLOADS_PER_STEP, PAYLOAD_BITS), device=device).float()
    clip_codeword = payload_to_codeword(payload)
    frame_symbols, segment_targets, slot_targets = temporal_frame_targets(
        clip_codeword, frames_per_payload
    )
    original = base_clip.repeat(PAYLOADS_PER_STEP, 1, 1, 1)
    baseline, watermarked, clean_latent, watermarked_latent, delta = latent_training_forward(
        original, frame_symbols
    )

    if stage['extract_domain'] == 'quantized_latent':
        latent_noise = torch.empty_like(clean_latent).uniform_(-0.5, 0.5)
        positive_latent = watermarked_latent + latent_noise
        negative_latent = clean_latent + latent_noise
        auxiliary_payload_loss = torch.zeros((), device=device)
    else:
        attack_index = (
            state["global_epoch"] * STEPS_PER_EPOCH + step_index
        ) % len(stage['attacks'])
        attack = stage['attacks'][attack_index]
        positive = apply_attack(watermarked, attack)
        negative = apply_attack(baseline, attack)
        positive_latent = CORE_CODEC.g_a(positive)
        negative_latent = CORE_CODEC.g_a(negative)

        # Auxiliary direct-latent objective mencegah carrier payload dilupakan
        # ketika model mulai menghadapi kompresi RGB yang berat.
        direct_hidden = latent_extractor.encode_features(watermarked_latent)
        direct_segments, direct_slots, _ = latent_extractor.classify(direct_hidden)
        auxiliary_payload_loss = (
            F.binary_cross_entropy_with_logits(direct_segments, segment_targets) +
            0.25 * F.cross_entropy(direct_slots, slot_targets)
        )

    positive_hidden = latent_extractor.encode_features(positive_latent)
    negative_hidden = latent_extractor.encode_features(negative_latent)
    segment_logits, slot_logits, frame_presence = latent_extractor.classify(positive_hidden)
    _, _, negative_frame_presence = latent_extractor.classify(negative_hidden)

    ordered_code_logits = aggregate_training_logits(segment_logits, PAYLOADS_PER_STEP, frames_per_payload)
    clip_presence = frame_presence.view(PAYLOADS_PER_STEP, frames_per_payload).mean(dim=1)
    negative_clip_presence = negative_frame_presence.view(
        PAYLOADS_PER_STEP, frames_per_payload
    ).mean(dim=1)

    payload_loss = (
        0.75 * F.binary_cross_entropy_with_logits(segment_logits, segment_targets) +
        1.25 * F.binary_cross_entropy_with_logits(ordered_code_logits, clip_codeword)
    )
    slot_loss = F.cross_entropy(slot_logits, slot_targets)
    presence_loss = 0.25 * (
        F.binary_cross_entropy_with_logits(frame_presence, torch.ones_like(frame_presence)) +
        F.binary_cross_entropy_with_logits(
            negative_frame_presence, torch.zeros_like(negative_frame_presence)
        ) +
        F.binary_cross_entropy_with_logits(clip_presence, torch.ones_like(clip_presence)) +
        F.binary_cross_entropy_with_logits(
            negative_clip_presence, torch.zeros_like(negative_clip_presence)
        )
    )

    incremental_mse = F.mse_loss(watermarked, baseline)
    incremental_ssim = differentiable_ssim(
        watermarked, baseline, data_range=1.0, size_average=True
    )
    mse_penalty = F.relu(
        incremental_mse - stage['quality_target_mse']
    ) / stage['quality_target_mse']
    ssim_penalty = F.relu(
        stage['quality_target_ssim'] - incremental_ssim
    ) / max(1.0 - stage['quality_target_ssim'], 0.05)
    quality_loss = mse_penalty + ssim_penalty
    latent_energy = delta.pow(2).mean()

    # Confidence-margin membuat extractor bukan hanya menebak sisi threshold
    # yang benar, tetapi menghasilkan logit dengan margin cukup besar untuk
    # bertahan setelah H.264/H.265/Neural compression.
    signed_code = clip_codeword * 2.0 - 1.0
    requested_margin = float(stage.get('logit_margin', 0.0))
    if requested_margin > 0:
        margin_loss = F.relu(
            requested_margin - signed_code * ordered_code_logits
        ).mean()
    else:
        margin_loss = torch.zeros((), device=device)

    loss = (
        stage.get('payload_weight', 2.0) * payload_loss +
        stage['slot_weight'] * slot_loss +
        stage['presence_weight'] * presence_loss +
        stage.get('auxiliary_weight', 0.25) * auxiliary_payload_loss +
        stage.get('margin_weight', 0.0) * margin_loss +
        stage['quality_weight'] * quality_loss +
        0.01 * latent_energy
    )

    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    trainable_parameters = [
        parameter
        for parameter in (
            list(latent_embedder.parameters()) +
            list(latent_extractor.parameters())
        )
        if parameter.requires_grad
    ]
    torch.nn.utils.clip_grad_norm_(trainable_parameters, 5.0)
    optimizer.step()

    with torch.no_grad():
        totals['loss'] += loss.item()
        totals['raw_code_bit_acc'] += (
            (ordered_code_logits > 0) == clip_codeword.bool()
        ).float().mean().item()
        totals['slot_accuracy'] += (
            slot_logits.argmax(dim=1) == slot_targets
        ).float().mean().item()
        totals['presence_tpr'] += float((torch.sigmoid(clip_presence) >= 0.5).float().mean())
        totals['negative_rejection'] += float((torch.sigmoid(negative_clip_presence) < 0.5).float().mean())
        totals['incremental_mse'] += incremental_mse.item()
        totals['incremental_ssim'] += incremental_ssim.item()
        totals['overall_mse'] += F.mse_loss(watermarked, original).item()

    state['totals'] = dict(totals)


def combine_validation(rows, stage):
    metrics = {key: float(np.mean([r[key] for r in rows]))
               for key in rows[0] if not key.startswith('_')}
    positives = [v for r in rows for v in r['_positive_scores']]
    negatives = [v for r in rows for v in r['_negative_scores']]
    threshold, tpr, rejection = calibrated_stage_presence(positives, negatives, stage['min_negative'])
    metrics.update(presence_threshold=threshold, presence_tpr=tpr, negative_rejection=rejection)
    return metrics


def validation_score(metrics):
    # Worst codec counts most; fixed validation samples across rounds.
    return (4 * metrics['post_ecc_exact_recovery'] + metrics['raw_code_bit_acc'] +
            metrics['presence_tpr'] + metrics['negative_rejection'] +
            0.05 * min(metrics['incremental_watermark_psnr'], 34.0) +
            min(metrics['incremental_watermark_ssim'], 0.94))

run_started = time.monotonic()
epochs_this_run = 0
budget_hit = False

def time_available():
    return (time.monotonic() - run_started) / 60 < MAX_TRAINING_MINUTES_PER_RUN

while RUN_MODE == 'train' and not state['stopped'] and state['stage'] < len(TRAINING_STAGES):
    stage = TRAINING_STAGES[state['stage']]
    if not time_available():
        budget_hit = True
        break
    if state['pending_validation']:
        # Save each completed video/codec job; resumed sessions do not restart the sweep.
        paths = deterministic_validation_paths(VALIDATION_VIDEOS_PER_CODEC, 0)
        for attack in stage['validation_attacks']:
            label = f'{attack[0]}_{attack[1]}'
            for path in paths:
                key = label + ':' + str(path.relative_to(DATASET_ROOT))
                if key in state['validation_jobs']:
                    continue
                if not time_available():
                    budget_hit = True
                    break
                print(f"Validation {label}: {path.name}", flush=True)
                single_stage = dict(stage, validation_attacks=[attack])
                state['validation_jobs'][key] = quick_validation(
                    single_stage, validation_round=0, paths_override=[path])
                save_training()
            if budget_hit:
                break
        if budget_hit:
            break
        by_codec = {}
        for attack in stage['validation_attacks']:
            label = f'{attack[0]}_{attack[1]}'
            rows = [v for key, v in state['validation_jobs'].items() if key.startswith(label + ':')]
            metrics = combine_validation(rows, stage)
            by_codec[label] = metrics
            state['codec_history'].append({'global_epoch': state['global_epoch'],
                'stage': stage['name'], 'codec': label, **metrics,
                'raw_BER': 1.0 - metrics['raw_code_bit_acc']})
        print(pd.DataFrame(by_codec).T[['raw_code_bit_acc', 'post_ecc_exact_recovery',
            'frame_slot_accuracy', 'presence_tpr', 'incremental_watermark_psnr',
            'incremental_watermark_ssim']].to_string())
        summary = {key: min(m[key] for m in by_codec.values()) for key in next(iter(by_codec.values()))}
        score = min(validation_score(m) for m in by_codec.values())
        improved = state['best_score'] is None or score > state['best_score'] + EARLY_STOP_MIN_DELTA
        passed = state['epoch'] >= stage['min_epochs'] and all(stage_passed(stage, m) for m in by_codec.values())
        state['pass_streak'] = state['pass_streak'] + 1 if passed else 0
        if improved or passed:
            state['best_score'] = score
            state['bad_rounds'] = 0
            atomic_save(snapshot(summary), best_path_for(state['stage']))
        else:
            state['bad_rounds'] += 1
        state['pending_validation'] = False
        state['validation_jobs'] = {}
        reason = None
        if state['pass_streak'] >= STAGE_PASS_PATIENCE:
            reason = 'passed'
        elif state['epoch'] >= stage['max_epochs']:
            reason = 'max_epochs_failed'
        elif not passed and state['epoch'] >= stage['min_epochs'] and state['bad_rounds'] >= EARLY_STOP_ROUNDS:
            reason = 'early_stop_plateau'
        if reason:
            state['records'].append({'stage_index': state['stage'] + 1,
                'stage_name': stage['name'], 'passed': reason == 'passed',
                'transition_reason': reason, 'epochs_used': state['epoch']})
            if reason != 'passed':
                state['stopped'] = True
                state['reason'] = reason + ': ' + stage['name']
            else:
                state['stage'] += 1
                state.update(epoch=0, best_score=None, bad_rounds=0, pass_streak=0)
                if state['stage'] < len(TRAINING_STAGES):
                    optimizer = make_optimizer(TRAINING_STAGES[state['stage']])
                    atomic_save(snapshot(summary), best_path_for(state['stage']))
        save_training()
        continue
    if epochs_this_run >= MAX_TRAINING_EPOCHS_PER_RUN:
        budget_hit = True
        break
    progress = state['epoch'] / max(stage['max_epochs'] - 1, 1)
    latent_embedder.strength = stage['strength_start'] + min(progress, 1) * (stage['strength_end'] - stage['strength_start'])
    latent_embedder.train(); latent_extractor.train()
    for step_index in tqdm(range(state['step'], STEPS_PER_EPOCH),
                           desc=f"{stage['name']} epoch {state['epoch'] + 1}/{stage['max_epochs']}"):
        if not time_available():
            budget_hit = True
            break
        train_one_step(stage, step_index)
        state['step'] = step_index + 1
        if state['step'] % CHECKPOINT_EVERY_STEPS == 0:
            save_training()
    if state['step'] < STEPS_PER_EPOCH:
        save_training()
        break
    state['epoch'] += 1
    state['global_epoch'] += 1
    epochs_this_run += 1
    state['history'].append({'global_epoch': state['global_epoch'], 'stage': stage['name'],
        'stage_epoch': state['epoch'], **{k: v / STEPS_PER_EPOCH for k, v in state['totals'].items()}})
    state.update(step=0, totals={})
    state['pending_validation'] = state['epoch'] % VALIDATE_EVERY == 0 or state['epoch'] == stage['max_epochs']
    print(state['history'][-1])
    save_training()
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

# Save training state before loading best weights for inference; never overwrite optimizer with best weights.
if RUN_MODE == 'train':
    save_training()
best_index = min(state['stage'], len(TRAINING_STAGES) - 1)
best = load_local_checkpoint(best_path_for(best_index))
latent_embedder.load_state_dict(best['latent_embedder'])
latent_extractor.load_state_dict(best['latent_extractor'])
latent_embedder.strength = best['latent_strength']
latent_embedder.eval(); latent_extractor.eval()
TRAINING_HIGHEST_STAGE = best_index
TRAINING_COMPLETED_ALL_STAGES = state['stage'] >= len(TRAINING_STAGES)
TRAINING_FINAL_STAGE_PASSED = TRAINING_COMPLETED_ALL_STAGES and all(r['passed'] for r in state['records'])
TRAINING_STOPPED_BY_GATE = state['stopped']
TRAINING_STOPPED_BY_RUNTIME_BUDGET = budget_hit
MODEL_RUN_TAG = f"{config_hash}_e{best['global_epoch']}"
training_status = {'completed_all_stages': TRAINING_COMPLETED_ALL_STAGES,
    'final_stage_passed': TRAINING_FINAL_STAGE_PASSED, 'stopped': state['stopped'],
    'reason': state['reason'], 'runtime_budget_hit': budget_hit,
    'global_epoch': state['global_epoch'], 'stage': state['stage'],
    'stage_epoch': state['epoch'], 'next_step': state['step'],
    'pending_validation': state['pending_validation'],
    'completed_validation_jobs': len(state['validation_jobs']),
    'best_validation': best['validation'], 'model_run_tag': MODEL_RUN_TAG}
pd.DataFrame(state['history']).to_csv(REPORT_DIR / 'training_history.csv', index=False)
pd.DataFrame(state['records']).to_csv(REPORT_DIR / 'training_stage_records.csv', index=False)
pd.DataFrame(state['codec_history']).to_csv(REPORT_DIR / 'validation_by_codec.csv', index=False)
(REPORT_DIR / 'training_status.json').write_text(json.dumps(training_status, indent=2))
print(json.dumps(training_status, indent=2))
print('Final stage lulus:', TRAINING_FINAL_STAGE_PASSED)
print('Laporan training:', REPORT_DIR)
if state['stopped']:
    print('Training berhenti permanen untuk konfigurasi ini; tinjau validation_by_codec.csv sebelum perubahan berikutnya.')
elif budget_hit:
    print('Checkpoint tersimpan. Jalankan ulang bagian 1–8 untuk melanjutkan step/validasi yang tertunda.')


In [ ]:
# 9. Evaluasi codec, latent extraction, dan perceptual video integrity
TEST_CODEC_CONFIGS = (
    [('H264', crf) for crf in (35, 40)] +
    [('H265', crf) for crf in (35, 40)] +
    [('NEURAL', quality) for quality in (1, 3)]
)
CALIBRATION_CONFIGS = TEST_CODEC_CONFIGS.copy()

def codec_roundtrip(frames, fps, codec_name, level, output_base):
    output_base = Path(output_base)
    if codec_name in ('H264', 'H265'):
        path = output_base.with_suffix('.mp4')
        reusable = (
            REUSE_EXISTING_EVAL_BITSTREAMS and path.exists() and path.stat().st_size > 0
        )
        if not reusable:
            codec = 'libx264' if codec_name == 'H264' else 'libx265'
            ffmpeg_encode(frames, path, fps, codec, level)
        try:
            reconstructed = read_encoded_video(path, len(frames))
        except Exception:
            codec = 'libx264' if codec_name == 'H264' else 'libx265'
            ffmpeg_encode(frames, path, fps, codec, level)
            reconstructed = read_encoded_video(path, len(frames))
    elif codec_name == 'NEURAL':
        path = output_base.with_suffix('.nvc')
        reusable = (
            REUSE_EXISTING_EVAL_BITSTREAMS and path.exists() and path.stat().st_size > 0
        )
        if not reusable:
            plain_neural_encode(frames, path, fps, level)
        try:
            reconstructed, _ = plain_neural_decode(path)
        except Exception:
            plain_neural_encode(frames, path, fps, level)
            reconstructed, _ = plain_neural_decode(path)
    else:
        raise ValueError(codec_name)
    return reconstructed, path, path.stat().st_size

def mean_quality(reference, candidate):
    count = min(len(reference), len(candidate))
    psnr_values, ssim_values = [], []
    for first, second in zip(reference[:count], candidate[:count]):
        psnr_values.append(peak_signal_noise_ratio(first, second, data_range=1.0))
        ssim_values.append(structural_similarity(
            first, second, data_range=1.0, channel_axis=-1
        ))
    return float(np.mean(psnr_values)), float(np.mean(ssim_values))

def neural_baseline_roundtrip(frames, fps=25.0):
    with tempfile.TemporaryDirectory() as tmp:
        path = Path(tmp) / 'baseline.nvc'
        plain_neural_encode(frames, path, fps, CORE_NEURAL_QUALITY)
        reconstructed, _ = plain_neural_decode(path)
        size = path.stat().st_size
    return reconstructed, size

def make_case_frames(original, case_name, fps=25.0):
    if case_name == 'no_watermark':
        frames, size = neural_baseline_roundtrip(original, fps)
        return frames, None, size
    text = TARGET_TEXT if case_name == 'target' else CONTROL_TEXT
    payload = text_to_payload(text)
    codeword = payload_to_codeword(payload)
    with tempfile.TemporaryDirectory() as tmp:
        path = Path(tmp) / 'watermarked.lwm'
        frames, _ = latent_watermark_encode(original, path, fps, payload)
        size = path.stat().st_size
    return frames, {
        'payload': payload.cpu().numpy().reshape(-1).astype(np.uint8),
        'codeword': codeword.cpu().numpy().reshape(-1).astype(np.uint8),
    }, size

def prediction_from_frames(frames, chunk_size=CODEC_BATCH_SIZE):
    segment_chunks, slot_chunks, presence_chunks = [], [], []
    with torch.inference_mode():
        for start in range(0, len(frames), chunk_size):
            tensor = frames_to_tensor(frames[start:start + chunk_size])
            latent = CORE_CODEC.g_a(tensor)
            hidden = latent_extractor.encode_features(latent)
            segment_logits, slot_logits, presence_logits = latent_extractor.classify(hidden)
            segment_chunks.append(segment_logits.cpu())
            slot_chunks.append(slot_logits.cpu())
            presence_chunks.append(presence_logits.cpu())
            del tensor, latent, hidden, segment_logits, slot_logits, presence_logits
        predicted, probabilities, presence, slot_coverage, slot_confidence = (
            aggregate_temporal_predictions(
                torch.cat(segment_chunks), torch.cat(slot_chunks), torch.cat(presence_chunks)
            )
        )
    outcome = decode_codeword_soft(probabilities)
    outcome['slot_coverage'] = slot_coverage
    outcome['slot_confidence'] = slot_confidence
    outcome['model_presence'] = presence
    effective_presence = presence  # CRC tetap diuji terpisah; jangan menimpa skor model.
    return outcome, predicted, probabilities, effective_presence
def payload_ber(expected, predicted):
    return float(np.mean(
        np.asarray(expected).reshape(-1) != np.asarray(predicted).reshape(-1)
    ))

def codeword_ber(expected, predicted):
    return float(np.mean(
        np.asarray(expected).reshape(-1) != np.asarray(predicted).reshape(-1)
    ))

def video_fingerprint(frames, sample_count=16):
    """Robust perceptual video fingerprint V10.3.

    Fingerprint lama hanya 64 DCT bits sehingga center-region tamper sering hanya
    mengubah 2-3 bit dan overlap dengan distorsi codec. Versi ini memakai temporal
    median + horizontal/vertical difference hash + low-frequency DCT.
    """
    frames = np.asarray(frames)
    indices = np.linspace(
        0, len(frames) - 1,
        min(sample_count, len(frames)),
        dtype=int,
    )

    grayscale = []
    for index in indices:
        frame = np.clip(frames[index] * 255.0, 0, 255).astype(np.uint8)
        gray = cv2.cvtColor(frame, cv2.COLOR_RGB2GRAY)
        gray = cv2.GaussianBlur(gray, (3, 3), 0)
        grayscale.append(
            cv2.resize(gray, (64, 64), interpolation=cv2.INTER_AREA).astype(np.float32)
        )

    temporal_median = np.median(np.stack(grayscale), axis=0).astype(np.float32)

    # 16x16 dHash horizontal: 256 bit
    h_grid = cv2.resize(
        temporal_median, (17, 16), interpolation=cv2.INTER_AREA
    )
    h_bits = (h_grid[:, 1:] >= h_grid[:, :-1]).reshape(-1)

    # 16x16 dHash vertical: 256 bit
    v_grid = cv2.resize(
        temporal_median, (16, 17), interpolation=cv2.INTER_AREA
    )
    v_bits = (v_grid[1:, :] >= v_grid[:-1, :]).reshape(-1)

    # Low-frequency pHash tambahan: 63 bit (DC dikeluarkan)
    dct_input = cv2.resize(
        temporal_median, (32, 32), interpolation=cv2.INTER_AREA
    )
    dct_low = cv2.dct(dct_input)[:8, :8].reshape(-1)
    dct_threshold = np.median(dct_low[1:])
    dct_bits = (dct_low[1:] >= dct_threshold).reshape(-1)

    return np.concatenate([h_bits, v_bits, dct_bits]).astype(np.uint8)

def fingerprint_hex(bits):
    return np.packbits(np.asarray(bits, dtype=np.uint8), bitorder='big').tobytes().hex()

def fingerprint_distance(first, second):
    return int(np.sum(
        np.asarray(first, dtype=np.uint8).reshape(-1) !=
        np.asarray(second, dtype=np.uint8).reshape(-1)
    ))

def tamper_frames(frames):
    tampered = np.asarray(frames).copy()
    height, width = tampered.shape[1:3]
    y0, y1 = height // 4, 3 * height // 4
    x0, x1 = width // 4, 3 * width // 4
    tampered[:, y0:y1, x0:x1] = 1.0 - tampered[:, y0:y1, x0:x1]
    return tampered

print(
    'Evaluasi V10.4 siap:', TEST_CODEC_CONFIGS,
    '| hanya H.264, H.265, dan neural codec.'
)


In [ ]:
# 10. Kalibrasi latent watermark pada validation split
RUN_FINAL_EVALUATION = bool(
    RUN_MODE == 'evaluate' and (TRAINING_FINAL_STAGE_PASSED or ALLOW_DIAGNOSTIC_EVALUATION)
)

if not RUN_FINAL_EVALUATION:
    print(
        '[SKIP EVALUASI] Final training stage belum lulus. '
        'Mode train tidak menjalankan test. Untuk evaluasi eksplisit gunakan RUN_MODE=evaluate; model belum lulus perlu ALLOW_DIAGNOSTIC_EVALUATION=True.'
    )
else:
    calibration_rows = []
    integrity_calibration_rows = []
    target_payload_np = text_to_payload(TARGET_TEXT).cpu().numpy().reshape(-1).astype(np.uint8)
    control_payload_np = text_to_payload(CONTROL_TEXT).cpu().numpy().reshape(-1).astype(np.uint8)
    target_codeword_np = payload_to_codeword(
        text_to_payload(TARGET_TEXT)
    ).cpu().numpy().reshape(-1).astype(np.uint8)

    with tempfile.TemporaryDirectory() as tmp:
        tmp = Path(tmp)
        for video_index, path in enumerate(tqdm(VAL_PATHS, desc='Calibration')):
            original, fps = load_eval_frames(path, max_frames=MAX_EVAL_FRAMES)
            original_fingerprint = video_fingerprint(original)
            for case_name in ('target', 'no_watermark', 'other_payload'):
                case_frames, own, core_bytes = make_case_frames(original, case_name, fps)
                for codec_name, level in CALIBRATION_CONFIGS:
                    base = tmp / f'{video_index}_{case_name}_{codec_name}_{level}'
                    reconstructed, _, _ = codec_roundtrip(case_frames, fps, codec_name, level, base)
                    outcome, predicted_code, _, presence = prediction_from_frames(reconstructed)
                    estimated = outcome['payload_bits'] if outcome['crc_valid'] else outcome['raw_payload_bits']
                    exact_target = bool(
                        outcome['crc_valid'] and
                        np.array_equal(outcome['payload_bits'], target_payload_np)
                    )
                    fp_distance = fingerprint_distance(
                        original_fingerprint, video_fingerprint(reconstructed)
                    )
                    calibration_rows.append({
                        'video': path.name, 'case': case_name,
                        'label_target': int(case_name == 'target'),
                        'codec': codec_name, 'level': level, 'presence': presence,
                        'ecc_success': bool(outcome['ecc_success']),
                        'crc_valid': bool(outcome['crc_valid']),
                        'exact_target_payload': exact_target,
                        'BER_to_target_payload': payload_ber(target_payload_np, estimated),
                        'raw_BER_to_target_codeword': codeword_ber(target_codeword_np, predicted_code),
                        'raw_own_payload_bit_acc': np.nan if own is None else 1.0 - payload_ber(
                            own['payload'], outcome['raw_payload_bits']
                        ),
                        'post_ecc_own_exact': False if own is None else bool(
                            outcome['crc_valid'] and np.array_equal(outcome['payload_bits'], own['payload'])
                        ),
                        'decoded_text': outcome['decoded_text'],
                        'slot_coverage': outcome['slot_coverage'],
                        'slot_confidence': outcome['slot_confidence'],
                        'fingerprint_distance': fp_distance,
                        'integrity_valid': fp_distance <= INTEGRITY_MAX_DISTANCE,
                        'core_bitstream_bytes': core_bytes,
                    })
                    if case_name == 'target':
                        integrity_calibration_rows.append({
                            'video': path.name, 'codec': codec_name, 'level': level,
                            'case': 'valid_target', 'expected_integrity_label': 1,
                            'fingerprint_distance': fp_distance,
                        })
                        tampered = tamper_frames(case_frames)
                        tampered_base = tmp / f'{video_index}_tampered_{codec_name}_{level}'
                        tampered_reconstructed, _, _ = codec_roundtrip(
                            tampered, fps, codec_name, level, tampered_base
                        )
                        integrity_calibration_rows.append({
                            'video': path.name, 'codec': codec_name, 'level': level,
                            'case': 'tampered_target', 'expected_integrity_label': 0,
                            'fingerprint_distance': fingerprint_distance(
                                original_fingerprint, video_fingerprint(tampered_reconstructed)
                            ),
                        })

    calibration_df = pd.DataFrame(calibration_rows)
    integrity_calibration_df = pd.DataFrame(integrity_calibration_rows)

    def calibrate_presence_threshold(frame, max_fpr=0.05):
        labels = frame['label_target'].to_numpy().astype(bool)
        no_watermark = frame['case'].to_numpy() == 'no_watermark'
        other_payload = frame['case'].to_numpy() == 'other_payload'
        exact = frame['exact_target_payload'].to_numpy().astype(bool)
        best_choice = None
        for threshold in np.r_[np.linspace(0.05, 0.995, 40), 1.001]:
            predicted = (frame['presence'].to_numpy() >= threshold) & exact
            tp = int(np.sum(predicted & labels)); fn = int(np.sum(~predicted & labels))
            fp = int(np.sum(predicted & ~labels)); tn = int(np.sum(~predicted & ~labels))
            tpr = tp / max(tp + fn, 1); fpr = fp / max(fp + tn, 1)
            no_wm_fpr = predicted[no_watermark].mean()
            other_fpr = predicted[other_payload].mean()
            if no_wm_fpr <= max_fpr and other_fpr <= max_fpr:
                candidate = (tpr, -max(no_wm_fpr, other_fpr), -fpr, threshold)
                if best_choice is None or candidate > best_choice[0]:
                    best_choice = (candidate, threshold, tpr, fpr, no_wm_fpr, other_fpr)
        return best_choice[1:]

    (PRESENCE_THRESHOLD, val_tpr, val_fpr,
     calibrated_no_wm_fpr, calibrated_other_fpr) = calibrate_presence_threshold(calibration_df)

    def calibrate_integrity_threshold(frame, max_tamper_fpr=0.05):
        labels = frame['expected_integrity_label'].to_numpy().astype(bool)
        distances = frame['fingerprint_distance'].to_numpy()
        best_choice = None
        # Fingerprint V10.3 lebih panjang dari 64 bit; grid threshold harus
        # mengikuti panjang fingerprint aktual, bukan hard-coded 0..64.
        max_distance = int(np.max(distances)) if len(distances) else 0
        for threshold in range(-1, max_distance + 2):
            predicted_valid = distances <= threshold
            tp = int(np.sum(predicted_valid & labels))
            fn = int(np.sum(~predicted_valid & labels))
            fp = int(np.sum(predicted_valid & ~labels))
            tn = int(np.sum(~predicted_valid & ~labels))
            tpr = tp / max(tp + fn, 1)
            fpr = fp / max(fp + tn, 1)
            if fpr <= max_tamper_fpr:
                candidate = (tpr, -fpr, threshold)
                if best_choice is None or candidate > best_choice[0]:
                    best_choice = (candidate, threshold, tpr, fpr)
        if best_choice is None:
            raise RuntimeError('Tidak ada ambang integritas yang memenuhi batas false accept.')
        return best_choice[1:]

    INTEGRITY_MAX_DISTANCE, integrity_val_tpr, integrity_val_fpr = calibrate_integrity_threshold(
        integrity_calibration_df
    )
    calibration_df['detected_as_sabila'] = (
        (calibration_df['presence'] >= PRESENCE_THRESHOLD) &
        calibration_df['exact_target_payload']
    )
    calibration_df['integrity_valid'] = (
        calibration_df['fingerprint_distance'] <= INTEGRITY_MAX_DISTANCE
    )

    target_rows = calibration_df.query("case == 'target'")
    other_rows = calibration_df.query("case == 'other_payload'")
    target_exact = target_rows['post_ecc_own_exact'].mean()
    other_exact = other_rows['post_ecc_own_exact'].mean()
    no_wm_fpr = calibration_df.query("case == 'no_watermark'")['detected_as_sabila'].mean()
    other_target_fpr = other_rows['detected_as_sabila'].mean()
    integrity_calibration_df['integrity_valid'] = (
        integrity_calibration_df['fingerprint_distance'] <= INTEGRITY_MAX_DISTANCE
    )

    diagnostic_df = calibration_df.groupby(['codec', 'level', 'case']).agg(
        presence_mean=('presence', 'mean'), crc_valid_rate=('crc_valid', 'mean'),
        slot_coverage_mean=('slot_coverage', 'mean'),
        slot_confidence_mean=('slot_confidence', 'mean'),
        post_ecc_exact_rate=('post_ecc_own_exact', 'mean'),
        fingerprint_distance_mean=('fingerprint_distance', 'mean'),
        integrity_valid_rate=('integrity_valid', 'mean'),
        detected_as_sabila_rate=('detected_as_sabila', 'mean'),
    ).reset_index()

    print(f'PRESENCE_THRESHOLD={PRESENCE_THRESHOLD:.3f}')
    print(f'Validation TPR/FPR: {val_tpr:.3f}/{val_fpr:.3f}')
    print(f'Exact sabila/control: {target_exact:.3f}/{other_exact:.3f}')
    print(f'No-watermark/other-payload FPR: {no_wm_fpr:.3f}/{other_target_fpr:.3f}')
    print(
        f'Integrity threshold={INTEGRITY_MAX_DISTANCE}; '
        f'validation TPR/FPR={integrity_val_tpr:.3f}/{integrity_val_fpr:.3f}'
    )
    display(diagnostic_df)

    gate_passed = (
        TRAINING_COMPLETED_ALL_STAGES and
        TRAINING_FINAL_STAGE_PASSED and
        TRAINING_HIGHEST_STAGE == len(TRAINING_STAGES) - 1 and
        val_tpr >= 0.80 and val_fpr <= 0.05 and
        target_exact >= 0.80 and other_exact >= 0.80 and
        no_wm_fpr <= 0.05 and other_target_fpr <= 0.05 and
        integrity_val_tpr >= 0.90 and integrity_val_fpr <= 0.05
    )
    calibration_df.to_csv(REPORT_DIR / 'validation_calibration.csv', index=False)
    integrity_calibration_df.to_csv(
        REPORT_DIR / 'validation_integrity_calibration.csv', index=False
    )
    diagnostic_df.to_csv(REPORT_DIR / 'validation_diagnostic_by_codec.csv', index=False)
    with (REPORT_DIR / 'calibrated_thresholds.json').open('w') as handle:
        json.dump({
            'presence_threshold': PRESENCE_THRESHOLD,
            'integrity_max_fingerprint_distance': INTEGRITY_MAX_DISTANCE,
            'convolutional_fec_required': True, 'crc6_required': True,
            'validation_tpr': val_tpr, 'validation_fpr': val_fpr,
            'integrity_validation_tpr': integrity_val_tpr,
            'integrity_validation_fpr': integrity_val_fpr,
        }, handle, indent=2)
    print('VALIDATION GATE:', 'LULUS' if gate_passed else 'GAGAL — final test hanya diagnostik')


In [ ]:
# 11. Final test: latent watermark, codec attacks, dan tamper controls
if not RUN_FINAL_EVALUATION:
    print('[SKIP FINAL TEST] Menunggu final training stage lulus.')
else:
    partial_detailed_path = REPORT_DIR / f'test_detailed_partial_{MODEL_RUN_TAG}.csv'
    partial_imperceptibility_path = REPORT_DIR / f'imperceptibility_partial_{MODEL_RUN_TAG}.csv'
    partial_fingerprint_path = REPORT_DIR / f'fingerprint_partial_{MODEL_RUN_TAG}.csv'

    detailed_rows = (
        pd.read_csv(partial_detailed_path).to_dict('records')
        if partial_detailed_path.exists() else []
    )
    imperceptibility_rows = (
        pd.read_csv(partial_imperceptibility_path).to_dict('records')
        if partial_imperceptibility_path.exists() else []
    )
    fingerprint_manifest = (
        pd.read_csv(partial_fingerprint_path).to_dict('records')
        if partial_fingerprint_path.exists() else []
    )
    expected_rows_per_video = 4 * len(TEST_CODEC_CONFIGS)
    completed_counts = pd.Series(
        [row['video'] for row in detailed_rows], dtype='object'
    ).value_counts().to_dict()
    completed_videos = {
        name for name, count in completed_counts.items()
        if int(count) == expected_rows_per_video
    }

    for video_index, path in enumerate(tqdm(TEST_PATHS, desc='Final test')):
        if path.name in completed_videos:
            print(f'[SKIP SELESAI] {path.name}')
            continue
        original, fps = load_eval_frames(path)
        original_fingerprint = video_fingerprint(original)
        safe_stem = re.sub(r'[^A-Za-z0-9_.-]+', '_', path.stem)
        run_stem = f'{safe_stem}_{MODEL_RUN_TAG}'

        baseline_path = BITSTREAM_DIR / f'{run_stem}_core_baseline.nvc'
        if not (
            REUSE_EXISTING_EVAL_BITSTREAMS and baseline_path.exists() and
            baseline_path.stat().st_size > 0
        ):
            plain_neural_encode(original, baseline_path, fps, CORE_NEURAL_QUALITY)
        baseline_frames, _ = plain_neural_decode(baseline_path)

        target_payload = text_to_payload(TARGET_TEXT)
        target_core_path = BITSTREAM_DIR / f'{run_stem}_target_latent.lwm'
        if (
            REUSE_EXISTING_EVAL_BITSTREAMS and target_core_path.exists() and
            target_core_path.stat().st_size > 0
        ):
            target_frames, _ = latent_watermark_decode(target_core_path)
        else:
            target_frames, _ = latent_watermark_encode(
                original, target_core_path, fps, target_payload
            )
        target_own = {
            'payload': target_payload.cpu().numpy().reshape(-1).astype(np.uint8),
            'codeword': payload_to_codeword(target_payload).cpu().numpy().reshape(-1).astype(np.uint8),
        }

        control_payload = text_to_payload(CONTROL_TEXT)
        control_core_path = BITSTREAM_DIR / f'{run_stem}_control_latent.lwm'
        if (
            REUSE_EXISTING_EVAL_BITSTREAMS and control_core_path.exists() and
            control_core_path.stat().st_size > 0
        ):
            control_frames, _ = latent_watermark_decode(control_core_path)
        else:
            control_frames, _ = latent_watermark_encode(
                original, control_core_path, fps, control_payload
            )
        control_own = {
            'payload': control_payload.cpu().numpy().reshape(-1).astype(np.uint8),
            'codeword': payload_to_codeword(control_payload).cpu().numpy().reshape(-1).astype(np.uint8),
        }
        tampered_frames = tamper_frames(target_frames)

        original_to_baseline_psnr, original_to_baseline_ssim = mean_quality(
            original, baseline_frames
        )
        original_to_target_psnr, original_to_target_ssim = mean_quality(
            original, target_frames
        )
        incremental_psnr, incremental_ssim = mean_quality(baseline_frames, target_frames)
        raw_bytes = len(original) * original.shape[1] * original.shape[2] * 3
        imperceptibility_rows.append({
            'video': path.name,
            'PSNR_original_vs_neural_baseline': original_to_baseline_psnr,
            'SSIM_original_vs_neural_baseline': original_to_baseline_ssim,
            'PSNR_original_vs_latent_watermarked': original_to_target_psnr,
            'SSIM_original_vs_latent_watermarked': original_to_target_ssim,
            'PSNR_neural_baseline_vs_latent_watermarked': incremental_psnr,
            'SSIM_neural_baseline_vs_latent_watermarked': incremental_ssim,
            'latent_core_bitstream_bytes': target_core_path.stat().st_size,
            'latent_core_compression_factor': raw_bytes / max(target_core_path.stat().st_size, 1),
        })
        fingerprint_manifest.append({
            'video': path.name, 'video_stem': path.stem,
            'reference_fingerprint_hex': fingerprint_hex(original_fingerprint),
        })

        cases = {
            'target': (target_frames, target_own, target_core_path.stat().st_size, 1),
            'no_watermark': (baseline_frames, None, baseline_path.stat().st_size, 1),
            'other_payload': (control_frames, control_own, control_core_path.stat().st_size, 1),
            'tampered_target': (tampered_frames, target_own, target_core_path.stat().st_size, 0),
        }

        for case_name, (case_frames, own, core_bytes, expected_integrity) in cases.items():
            for codec_name, level in TEST_CODEC_CONFIGS:
                output_base = BITSTREAM_DIR / f'{run_stem}_{case_name}_{codec_name.lower()}_{level}'
                reconstructed, bitstream_path, encoded_bytes = codec_roundtrip(
                    case_frames, fps, codec_name, level, output_base
                )
                outcome, predicted_code, _, presence = prediction_from_frames(reconstructed)
                estimated = outcome['payload_bits'] if outcome['crc_valid'] else outcome['raw_payload_bits']
                exact_target = bool(
                    outcome['crc_valid'] and np.array_equal(outcome['payload_bits'], target_payload_np)
                )
                detected = presence >= PRESENCE_THRESHOLD and exact_target
                fp_distance = fingerprint_distance(
                    original_fingerprint, video_fingerprint(reconstructed)
                )
                integrity_valid = fp_distance <= INTEGRITY_MAX_DISTANCE
                quality_psnr, quality_ssim = mean_quality(case_frames, reconstructed)
                n, h, w, _ = case_frames.shape
                duration = n / fps
                attack_raw_bytes = n * h * w * 3
                detailed_rows.append({
                    'video': path.name, 'split': 'test', 'case': case_name,
                    'true_target_label': int(case_name in ('target', 'tampered_target')),
                    'expected_integrity_label': int(expected_integrity),
                    'codec': codec_name, 'level': int(level), 'frames': n,
                    'resolution': f'{w}x{h}', 'duration_s': duration,
                    'core_latent_bitstream_bytes': core_bytes,
                    'attack_encoded_bytes': encoded_bytes,
                    'bpp': encoded_bytes * 8 / (n * h * w),
                    'bitrate_kbps': encoded_bytes * 8 / max(duration, 1e-9) / 1000,
                    'compression_factor_raw_over_encoded': attack_raw_bytes / max(encoded_bytes, 1),
                    'PSNR_input_vs_reconstructed': quality_psnr,
                    'SSIM_input_vs_reconstructed': quality_ssim,
                    'presence_probability': presence,
                    'ecc_success': bool(outcome['ecc_success']),
                    'crc_valid': bool(outcome['crc_valid']),
                    'corrected_symbols': outcome['corrected_symbols'],
                    'BER_to_sabila_48_payload_bits': payload_ber(target_payload_np, estimated),
                    'raw_BER_to_sabila_codeword': codeword_ber(target_codeword_np, predicted_code),
                    'detected_as_sabila': bool(detected),
                    'decoded_text': outcome['decoded_text'],
                    'raw_text_before_ecc': outcome['raw_text'],
                    'slot_coverage': outcome['slot_coverage'],
                    'slot_confidence': outcome['slot_confidence'],
                    'post_ecc_own_exact': False if own is None else bool(
                        outcome['crc_valid'] and np.array_equal(outcome['payload_bits'], own['payload'])
                    ),
                    'fingerprint_distance_to_original': fp_distance,
                    'integrity_valid': bool(integrity_valid),
                    'automatic_verification_passed': bool(detected and integrity_valid),
                    'attack_bitstream_file': str(bitstream_path),
                })

        # Commit setelah satu video lengkap. Jika runtime terputus, video ini tidak
        # perlu dikompresi ulang pada sesi berikutnya.
        pd.DataFrame(detailed_rows).to_csv(partial_detailed_path, index=False)
        pd.DataFrame(imperceptibility_rows).to_csv(
            partial_imperceptibility_path, index=False
        )
        pd.DataFrame(fingerprint_manifest).to_csv(partial_fingerprint_path, index=False)
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    detailed_df = pd.DataFrame(detailed_rows)
    imperceptibility_df = pd.DataFrame(imperceptibility_rows)
    fingerprint_manifest_df = pd.DataFrame(fingerprint_manifest)
    fingerprint_manifest_df.to_csv(REPORT_DIR / 'fingerprint_manifest.csv', index=False)
    print('Final test selesai:', len(detailed_df), 'sampel termasuk tamper controls.')


In [ ]:
# 12. Detection, recovery, integrity, quality, dan acceptance report
if not RUN_FINAL_EVALUATION:
    print('[SKIP REPORT] Belum ada final test V10.6 yang valid.')
else:
    def confusion_metrics(group, truth_column, prediction_column):
        truth = group[truth_column].astype(bool).to_numpy()
        pred = group[prediction_column].astype(bool).to_numpy()
        tp = int(np.sum(truth & pred)); fn = int(np.sum(truth & ~pred))
        fp = int(np.sum(~truth & pred)); tn = int(np.sum(~truth & ~pred))
        return {
            'TP': tp, 'TN': tn, 'FP': fp, 'FN': fn,
            'accuracy': (tp + tn) / max(tp + tn + fp + fn, 1),
            'precision': tp / max(tp + fp, 1),
            'recall_TPR': tp / max(tp + fn, 1),
            'specificity_TNR': tn / max(tn + fp, 1),
            'FPR': fp / max(fp + tn, 1),
        }

    def to_builtin_types(value):
        """Ubah scalar/container NumPy menjadi tipe bawaan yang aman untuk JSON."""
        if isinstance(value, np.generic):
            return value.item()
        if isinstance(value, dict):
            return {key: to_builtin_types(item) for key, item in value.items()}
        if isinstance(value, (list, tuple)):
            return [to_builtin_types(item) for item in value]
        return value

    detection_df = detailed_df.query("case != 'tampered_target'")
    summary_rows = []
    for (codec_name, level), group in detection_df.groupby(['codec', 'level'], sort=False):
        summary_rows.append({
            'codec': codec_name, 'level': level,
            **confusion_metrics(group, 'true_target_label', 'detected_as_sabila'),
        })
    summary_df = pd.DataFrame(summary_rows)

    target_recovery = detection_df.query("case == 'target'").groupby(
        ['codec', 'level']
    )['post_ecc_own_exact'].mean().rename('exact_sabila_recovery').reset_index()
    control_recovery = detection_df.query("case == 'other_payload'").groupby(
        ['codec', 'level']
    )['post_ecc_own_exact'].mean().rename('exact_control_recovery').reset_index()
    compression_summary = detection_df.query("case == 'target'").groupby(
        ['codec', 'level']
    )[['bpp', 'bitrate_kbps', 'compression_factor_raw_over_encoded',
       'PSNR_input_vs_reconstructed', 'SSIM_input_vs_reconstructed',
       'slot_coverage', 'slot_confidence']].mean().reset_index()
    summary_df = summary_df.merge(target_recovery, on=['codec', 'level']).merge(
        control_recovery, on=['codec', 'level']
    ).merge(compression_summary, on=['codec', 'level'])

    integrity_df = detailed_df.query("case in ['target', 'tampered_target']")
    integrity_metrics = confusion_metrics(
        integrity_df, 'expected_integrity_label', 'integrity_valid'
    )
    tamper_rejection = 1.0 - integrity_df.query(
        "case == 'tampered_target'"
    )['integrity_valid'].mean()

    high_compression = summary_df[
        ((summary_df.codec.isin(['H264', 'H265'])) & (summary_df.level >= 35)) |
        ((summary_df.codec == 'NEURAL') & (summary_df.level == 1))
    ]
    acceptance = {
        'latent_embedding_used': True,
        'temporal_interleaving_used': True,
        'mean_slot_coverage_at_least_7_of_8': bool(
            (summary_df['slot_coverage'] >= 7.0).all()
        ),
        'split_disjoint': True,
        'all_training_stages_executed': bool(TRAINING_COMPLETED_ALL_STAGES),
        'final_training_stage_passed': bool(TRAINING_FINAL_STAGE_PASSED),
        'validation_gate_passed': bool(gate_passed),
        'test_FPR_at_most_5_percent': bool((summary_df['FPR'] <= 0.05).all()),
        'high_compression_recall_at_least_80_percent': bool(
            (high_compression['recall_TPR'] >= 0.80).all()
        ),
        'control_payload_recovery_at_least_80_percent': bool(
            (summary_df['exact_control_recovery'] >= 0.80).all()
        ),
        'integrity_accuracy_at_least_90_percent': bool(integrity_metrics['accuracy'] >= 0.90),
        'tamper_rejection_at_least_90_percent': bool(tamper_rejection >= 0.90),
        'latent_bitstream_smaller_than_raw': bool(
            (imperceptibility_df['latent_core_compression_factor'] > 1.0).all()
        ),
        'incremental_watermark_PSNR_at_least_34dB': bool(
            imperceptibility_df['PSNR_neural_baseline_vs_latent_watermarked'].mean() >= 34.0
        ),
        'incremental_watermark_SSIM_at_least_0_94': bool(
            imperceptibility_df['SSIM_neural_baseline_vs_latent_watermarked'].mean() >= 0.94
        ),
    }
    acceptance['all_passed'] = all(acceptance.values())

    # Simpan margin terhadap target supaya FAIL berikutnya langsung menunjukkan
    # berapa jauh metrik dari acceptance threshold; threshold penelitian tidak diubah.
    acceptance_margins = {
        'high_compression_min_recall': float(high_compression['recall_TPR'].min()),
        'control_payload_min_recovery': float(summary_df['exact_control_recovery'].min()),
        'integrity_accuracy': float(integrity_metrics['accuracy']),
        'tamper_rejection': float(tamper_rejection),
        'incremental_watermark_PSNR_mean': float(
            imperceptibility_df['PSNR_neural_baseline_vs_latent_watermarked'].mean()
        ),
        'incremental_watermark_SSIM_mean': float(
            imperceptibility_df['SSIM_neural_baseline_vs_latent_watermarked'].mean()
        ),
    }
    acceptance = to_builtin_types(acceptance)

    detailed_df.to_csv(REPORT_DIR / 'test_detailed.csv', index=False)
    summary_df.to_csv(REPORT_DIR / 'test_summary.csv', index=False)
    imperceptibility_df.to_csv(REPORT_DIR / 'imperceptibility.csv', index=False)
    with (REPORT_DIR / 'integrity_metrics.json').open('w') as handle:
        json.dump(
            to_builtin_types({**integrity_metrics, 'tamper_rejection': tamper_rejection}),
            handle, indent=2,
        )
    with (REPORT_DIR / 'acceptance_report.json').open('w') as handle:
        json.dump(acceptance, handle, indent=2)
    with (REPORT_DIR / 'acceptance_margins.json').open('w') as handle:
        json.dump(to_builtin_types(acceptance_margins), handle, indent=2)

    display(summary_df)
    display(imperceptibility_df.describe())
    print('\nINTEGRITY METRICS:', integrity_metrics, 'tamper_rejection=', tamper_rejection)
    print('\nACCEPTANCE REPORT')
    print('METRIC MARGINS:', acceptance_margins)
    for criterion, passed in acceptance.items():
        print(f"{'LULUS' if passed else 'GAGAL'} | {criterion}")
    print(
        '\nSELURUH KRITERIA LULUS.' if acceptance['all_passed'] else
        '\nEksperimen selesai, tetapi hasil belum mendukung seluruh klaim.'
    )


In [ ]:
# 13. Visualisasi ringkas
if not RUN_FINAL_EVALUATION:
    print('[SKIP VISUALISASI] Belum ada final report V10.6.')
else:
    fig, axes = plt.subplots(1, 3, figsize=(16, 4))

    labels = summary_df['codec'] + '-' + summary_df['level'].astype(str)
    axes[0].bar(labels, summary_df['recall_TPR'])
    axes[0].axhline(0.80, color='red', linestyle='--')
    axes[0].set_title('Recall latent watermark setelah convolutional FEC/CRC-6')
    axes[0].tick_params(axis='x', rotation=60)
    axes[0].set_ylim(0, 1.05)

    axes[1].bar(labels, summary_df['FPR'])
    axes[1].axhline(0.05, color='red', linestyle='--')
    axes[1].set_title('False Positive Rate')
    axes[1].tick_params(axis='x', rotation=60)
    axes[1].set_ylim(0, max(0.1, summary_df['FPR'].max() * 1.2))

    axes[2].bar(labels, summary_df['bpp'])
    axes[2].set_title('Bit per pixel dari bitstream nyata')
    axes[2].tick_params(axis='x', rotation=60)

    plt.tight_layout()
    figure_path = REPORT_DIR / 'validated_summary.png'
    plt.savefig(figure_path, dpi=160, bbox_inches='tight')
    plt.show()
    print('Laporan tersimpan di:', REPORT_DIR)


In [ ]:
# 14. Tool verifikasi otomatis: upload → extract → integrity → report
if not RUN_FINAL_EVALUATION:
    print('[SKIP TOOL VERIFIKASI] Threshold final belum dikalibrasi.')
else:
    def uploaded_path(value):
        if value is None:
            return None
        if isinstance(value, dict):
            return value.get('path') or value.get('video') or value.get('name')
        return str(value)

    def verify_uploaded_video(video_upload, reference_upload, expected_text):
        video_path = uploaded_path(video_upload)
        reference_path = uploaded_path(reference_upload)
        if not video_path:
            return {'status': 'ERROR', 'message': 'Video yang diverifikasi belum dipilih.'}
        try:
            expected_payload = text_to_payload(expected_text)
        except Exception as error:
            return {'status': 'ERROR', 'message': str(error)}

        frames, _ = load_eval_frames(Path(video_path))
        outcome, _, _, presence = prediction_from_frames(frames)
        expected_bits = expected_payload.cpu().numpy().reshape(-1).astype(np.uint8)
        watermark_verified = bool(
            presence >= PRESENCE_THRESHOLD and outcome['crc_valid'] and
            np.array_equal(outcome['payload_bits'], expected_bits)
        )

        fingerprint_distance_value = None
        integrity_valid = None
        integrity_status = 'REFERENCE_REQUIRED'
        if reference_path:
            reference_frames, _ = load_eval_frames(Path(reference_path))
            fingerprint_distance_value = fingerprint_distance(
                video_fingerprint(reference_frames), video_fingerprint(frames)
            )
            integrity_valid = fingerprint_distance_value <= INTEGRITY_MAX_DISTANCE
            integrity_status = 'VALID' if integrity_valid else 'MODIFIED'

        if watermark_verified and integrity_valid is True:
            overall_status = 'VERIFIED'
        elif watermark_verified and integrity_valid is None:
            overall_status = 'WATERMARK_VALID_INTEGRITY_NOT_CHECKED'
        elif watermark_verified:
            overall_status = 'WATERMARK_VALID_BUT_CONTENT_MODIFIED'
        else:
            overall_status = 'WATERMARK_INVALID_OR_NOT_FOUND'

        report = {
            'status': overall_status,
            'expected_text': expected_text,
            'decoded_text': outcome['decoded_text'],
            'raw_text_before_ecc': outcome['raw_text'],
            'presence_probability': round(float(presence), 6),
            'slot_coverage': int(outcome['slot_coverage']),
            'slot_confidence': round(float(outcome['slot_confidence']), 6),
            'presence_threshold': round(float(PRESENCE_THRESHOLD), 6),
            'ecc_success': bool(outcome['ecc_success']),
            'crc_valid': bool(outcome['crc_valid']),
            'corrected_symbols': None if np.isnan(outcome['corrected_symbols']) else int(outcome['corrected_symbols']),
            'watermark_verified': watermark_verified,
            'integrity_status': integrity_status,
            'fingerprint_distance': fingerprint_distance_value,
            'integrity_max_distance': INTEGRITY_MAX_DISTANCE,
            'integrity_valid': integrity_valid,
            'payload_layout': f'{TEMPORAL_SLOTS}_slots_x_{BITS_PER_SLOT}_bits',
            'backbone': f'bmshj2018_factorized_quality_{CORE_NEURAL_QUALITY}_framewise',
        }
        with (REPORT_DIR / 'last_upload_verification.json').open('w') as handle:
            json.dump(report, handle, indent=2)
        return report

    verification_app = gr.Interface(
        fn=verify_uploaded_video,
        inputs=[
            gr.Video(label='Video yang diverifikasi', sources=['upload']),
            gr.Video(label='Video referensi untuk integritas (opsional)', sources=['upload']),
            gr.Textbox(value=TARGET_TEXT, label='Watermark yang diharapkan'),
        ],
        outputs=gr.JSON(label='Laporan verifikasi otomatis'),
        title='Temporal Latent Neural Watermark Verification',
        description=(
            'Merakit watermark dari slot temporal latent, memvalidasi convolutional FEC/CRC-6, '
            'dan membandingkan perceptual fingerprint jika video referensi diberikan.'
        ),
    )
    verification_app.launch(share=True, debug=False)


## Menjalankan V10.6

1. Upload notebook ini ke Colab, pilih GPU, jalankan bagian 1–8 (judul sel).
2. Default `RUN_MODE='train'`, 4 epoch / 45 menit per run. Batas waktu diperiksa di antara step dan pekerjaan validasi; satu pekerjaan aktif bisa melewati batas. FFmpeg memiliki timeout 120 detik.
3. Warm start membutuhkan `models/best_v105_stage4_5e07c20697.pth`. Jika namanya berbeda, isi `WARM_START_V105` dengan checkpoint milik eksperimenmu. Tidak ada fallback bobot acak.
4. Jalankan ulang bagian 1–8 untuk resume; step, optimizer, hasil validasi parsial, dan early-stop counter disimpan di `models/checkpoint_v106_<hash>.pth`.
5. Jika `stopped=True`, training berhenti permanen untuk konfigurasi tersebut. Tinjau `training_status.json` dan `validation_by_codec.csv`; rerun tidak mengulangi training gagal.
6. Lihat recovery dan BER pada validation, bukan test, untuk memilih perubahan model. Subset tetap 6 video/codec adalah gate development; bukan jaminan generalisasi.
7. Untuk laporan akhir: set `RUN_MODE='evaluate'`, jalankan bagian 1–14. Bila model belum lulus, set `ALLOW_DIAGNOSTIC_EVALUATION=True` untuk laporan diagnostik yang tetap mencatat GAGAL.
8. Laporan ada di `output/v9_2_temporal_latent_validated/reports/v10_6/<hash>/`. V10.5 tidak ditimpa. Ambang kualitas dan recovery tidak diturunkan.

Strength 0.40→0.34 dan curriculum baru adalah hipotesis perbaikan, bukan jaminan recovery 80%. Protokol frame V10.6 berubah ke frame berurutan; rerun baseline dengan protokol yang sama diperlukan untuk perbandingan yang adil. Test yang sudah dilihat menjadi diagnostik; gunakan held-out baru sebelum klaim generalisasi final.

Verifikasi lokal: sintaks semua sel, 128 kasus FEC bersih/rusak satu bit, agregasi batch 1/2 pada 8/16/64 frame, serta simulasi resume step/validasi dan early stop lulus. PyTorch/GPU tidak tersedia lokal; gradient dan training riil diperiksa lewat preflight Colab dan run berikutnya.
